In [ ]:
# In[1]:

import pandas as pd
import pytz
from pathlib import Path

tz = pytz.timezone('Asia/Shanghai')

# Load CSVs (store full tables in memory as variables)
metric_csv = Path("metric.csv")
trace_csv = Path("trace.csv")
log_csv = Path("log.csv")
error_csv = Path("error_logs.csv")

metric_df = pd.read_csv(metric_csv) if metric_csv.exists() else pd.DataFrame(columns=["timestamp","cmdb_id","kpi_name","value"])
trace_df = pd.read_csv(trace_csv) if trace_csv.exists() else pd.DataFrame(columns=["timestamp","cmdb_id","trace_name","value"])
log_df = pd.read_csv(log_csv) if log_csv.exists() else pd.DataFrame(columns=["timestamp","cmdb_id","log_name","value"])
error_df = pd.read_csv(error_csv) if error_csv.exists() else pd.DataFrame(columns=["timestamp","cmdb_id","message"])

# Convert timestamp -> timezone-aware datetime (Asia/Shanghai, UTC+8)
for df in (metric_df, trace_df, log_df, error_df):
    if 'timestamp' in df.columns and not df.empty:
        # timestamps are seconds, floored to minute
        df['dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        df['dt'] = pd.Series(dtype='datetime64[ns]')

# 1) metric.csv summary grouped by cmdb_id and kpi_name
if not metric_df.empty:
    metric_grp = metric_df.groupby(['cmdb_id','kpi_name'])['value']
    metric_summary = metric_grp.agg(
        count='count',
        p50=lambda s: s.quantile(0.5),
        p95=lambda s: s.quantile(0.95),
        min='min',
        max='max'
    ).reset_index()
    metric_summary_top50 = metric_summary.sort_values('count', ascending=False).head(50)
else:
    metric_summary = pd.DataFrame(columns=['cmdb_id','kpi_name','count','p50','p95','min','max'])
    metric_summary_top50 = metric_summary.copy()

# 2) trace.csv summary grouped by cmdb_id and trace_name
if not trace_df.empty:
    trace_grp = trace_df.groupby(['cmdb_id','trace_name'])['value']
    trace_summary = trace_grp.agg(
        count='count',
        p50=lambda s: s.quantile(0.5),
        p95=lambda s: s.quantile(0.95),
        min='min',
        max='max'
    ).reset_index()
    trace_summary_top50 = trace_summary.sort_values('count', ascending=False).head(50)
else:
    trace_summary = pd.DataFrame(columns=['cmdb_id','trace_name','count','p50','p95','min','max'])
    trace_summary_top50 = trace_summary.copy()

# 3) log.csv summary grouped by cmdb_id and log_name
if not log_df.empty:
    log_grp = log_df.groupby(['cmdb_id','log_name'])['value']
    log_summary = log_grp.agg(
        count='count',
        p50=lambda s: s.quantile(0.5),
        p95=lambda s: s.quantile(0.95),
        min='min',
        max='max'
    ).reset_index()
    log_summary_top50 = log_summary.sort_values('count', ascending=False).head(50)
else:
    log_summary = pd.DataFrame(columns=['cmdb_id','log_name','count','p50','p95','min','max'])
    log_summary_top50 = log_summary.copy()

# 4) error_logs.csv compact info: number of rows, unique cmdb_id list, earliest and latest timestamp
if not error_df.empty:
    error_rows_count = int(error_df.shape[0])
    error_unique_cmdb = sorted(error_df['cmdb_id'].dropna().unique().tolist())
    error_earliest = error_df['dt'].min()
    error_latest = error_df['dt'].max()
else:
    error_rows_count = 0
    error_unique_cmdb = []
    error_earliest = pd.NaT
    error_latest = pd.NaT

error_summary_compact = pd.DataFrame([{
    'rows_count': error_rows_count,
    'unique_cmdb_list': error_unique_cmdb,
    'unique_cmdb_count': len(error_unique_cmdb),
    'earliest_dt': error_earliest,
    'latest_dt': error_latest
}])

# 5) global unique cmdb_id across all files and overall earliest/latest timestamp across all telemetry files
all_cmdbs = set()
for df in (metric_df, trace_df, log_df, error_df):
    if 'cmdb_id' in df.columns:
        all_cmdbs.update(df['cmdb_id'].dropna().unique().tolist())
all_cmdbs = sorted(all_cmdbs)

# overall earliest/latest across all dt columns
combined_dts = pd.concat([df['dt'] for df in (metric_df, trace_df, log_df, error_df) if not df.empty], ignore_index=True)
overall_earliest = combined_dts.min() if not combined_dts.empty else pd.NaT
overall_latest = combined_dts.max() if not combined_dts.empty else pd.NaT

global_summary_compact = pd.DataFrame([{
    'unique_cmdb_list': all_cmdbs,
    'unique_cmdb_count': len(all_cmdbs),
    'overall_earliest_dt': overall_earliest,
    'overall_latest_dt': overall_latest,
    'total_rows_all_files': int(sum(df.shape[0] for df in (metric_df, trace_df, log_df, error_df)))
}])

# Final displayed outputs (compact). Variables hold full tables if needed.
metric_summary_top50, trace_summary_top50, log_summary_top50, error_summary_compact, global_summary_compact

```
Out[1]:
```
```python
# Compose a concise plain-English summary string based on previous results in the kernel.
# Uses existing variables: metric_summary_top50, trace_summary_top50, log_summary_top50,
# error_summary_compact, global_summary_compact

# Safely extract values (handle potential empties)
def safe_get(df, idx=0):
    return None if df.empty else df.iloc[idx]

top_metric = safe_get(metric_summary_top50)
top_trace = safe_get(trace_summary_top50)
top_log = safe_get(log_summary_top50)

total_rows = int(global_summary_compact.loc[0, 'total_rows_all_files'])
unique_cmdbs = global_summary_compact.loc[0, 'unique_cmdb_list']
unique_count = int(global_summary_compact.loc[0, 'unique_cmdb_count'])
overall_earliest = global_summary_compact.loc[0, 'overall_earliest_dt']
overall_latest = global_summary_compact.loc[0, 'overall_latest_dt']

error_rows = int(error_summary_compact.loc[0, 'rows_count'])
error_cmdbs = error_summary_compact.loc[0, 'unique_cmdb_list']

summary = (
    f"Telemetry summary (concise):\n"
    f"- Total telemetry rows (all files): {total_rows}\n"
    f"- Unique cmdb_id count: {unique_count}; examples: {unique_cmdbs[:10]}{(' ...' if len(unique_cmdbs)>10 else '')}\n"
    f"- Time range (UTC+8): {overall_earliest} to {overall_latest}\n"
    f"- error_logs.csv: {error_rows} rows; cmdb_ids: {error_cmdbs}\n\n"
    f"Top group examples (by count):\n"
)

if top_metric is not None:
    summary += (
        f"- Metric (example top group): cmdb_id={top_metric['cmdb_id']}, kpi_name={top_metric['kpi_name']}, "
        f"count={int(top_metric['count'])}, p50={top_metric['p50']}, p95={top_metric['p95']}, "
        f"min={top_metric['min']}, max={top_metric['max']}\n"
    )
else:
    summary += "- Metric: no data\n"

if top_trace is not None:
    summary += (
        f"- Trace (example top group): cmdb_id={top_trace['cmdb_id']}, trace_name={top_trace['trace_name']}, "
        f"count={int(top_trace['count'])}, p50={top_trace['p50']}, p95={top_trace['p95']}, "
        f"min={top_trace['min']}, max={top_trace['max']}\n"
    )
else:
    summary += "- Trace: no data\n"

if top_log is not None:
    summary += (
        f"- Log (example top group): cmdb_id={top_log['cmdb_id']}, log_name={top_log['log_name']}, "
        f"count={int(top_log['count'])}, p50={top_log['p50']}, p95={top_log['p95']}, "
        f"min={top_log['min']}, max={top_log['max']}\n"
    )
else:
    summary += "- Log: no data\n"

# Display the concise summary string
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                           kpi_name  count        p50        p95        min        max
6         IG01           JVM-Threads_7778_JVM_ThreadCount_Threads     31       72.0       76.0       68.0       77.0
1584  Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31        0.0        0.0        0.0        0.0
1585  Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31        0.0        0.0        0.0        0.0
1586  Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31        0.0        0.0        0.0        0.0
386       MG02                 OSLinux-OSLinux_ZABBIX_Host_Uptime     31  7866428.0  7867241.0  7865526.0  7867332.0
...        ...                                                ...    ...        ...        ...        ...        ...
1218  Tomcat01     Tomcat-Sessions_7441--_SESSIONRejectedSessions     31        0.0        0.0        0.0        0.0
1219  Tomcat01  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31        0.0        0.0        0.0        0.0
1220  Tomcat01  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31        0.0        0.0        0.0        0.0
1300  Tomcat02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT     31        0.0        0.0        0.0        0.0
1329  Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...     31   400206.0   400206.0   400206.0   400206.0

[50 rows x 7 columns],    cmdb_id                       trace_name  count          p50          p95        min          max
0     IG01         trace.self.duration_mean     31     0.245259     0.427851   0.166977     0.450443
1     IG01          trace.self.duration_p95     31     1.087550     2.089000   1.035000     2.101050
2     IG01             trace.self.row_count     31  1018.000000  1340.000000  12.000000  1374.000000
3     IG01  trace.to_Tomcat01.duration_mean     31     0.242208     0.390130   0.018000     0.441965
4     IG01   trace.to_Tomcat01.duration_p95     31     1.075900     2.100050   0.025200     2.115100
..     ...                              ...    ...          ...          ...        ...          ...
62    MG01      trace.to_dockerA2.row_count     31    98.000000   146.500000   3.000000   151.000000
63    MG01  trace.to_dockerB1.duration_mean     31     0.199113     0.532072   0.002500     0.612978
65    MG01      trace.to_dockerB1.row_count     31   168.000000   204.500000   4.000000   219.000000
56    MG01             trace.self.row_count     31   763.000000  1015.000000  10.000000  1087.000000
55    MG01          trace.self.duration_p95     31     1.027000     1.079500   0.120000     1.087150

[50 rows x 7 columns],      cmdb_id          log_name  count     p50      p95    min     max
4   Tomcat01     log.row_count     31  256.00  336.500   6.00  350.00
2   Tomcat01   log.error_count     31    0.00    0.000   0.00    0.00
16  apache02   log.error_count     31    0.00    0.000   0.00    0.00
15  apache01     log.row_count     31  522.00  660.500  10.00  670.00
5   Tomcat02   log.error_count     31    0.00    0.000   0.00    0.00
7   Tomcat02     log.row_count     31  267.00  340.500   2.00  353.00
8   Tomcat03   log.error_count     31    0.00    0.000   0.00    0.00
13  Tomcat04     log.row_count     31  252.00  324.500   4.00  357.00
11  Tomcat04   log.error_count     31    0.00    0.000   0.00    0.00
10  Tomcat03     log.row_count     31  272.00  331.500   3.00  377.00
17  apache02     log.row_count     31  518.00  656.000   5.00  705.00
14  apache01   log.error_count     31    0.00    0.000   0.00    0.00
1    Mysql01     log.row_count     29    5.00    9.600   1.00   13.00
0    Mysql01   log.error_count     29    0.00    0.000   0.00    0.00
12  Tomcat04  log.gc_pause_max     14    0.06    0.080   0.03    0.08
9   Tomcat03  log.gc_pause_max     13    0.07    0.084   0.03    0.09
3   Tomcat01  log.gc_pause_max     13    0.07    0.094   0.03    0.10
6   Tomcat02  log.gc_pause_max     13    0.07    0.088   0.03    0.10,    rows_count unique_cmdb_list  unique_cmdb_count earliest_dt latest_dt
0           0               []                  0         NaT       NaT,                                     unique_cmdb_list  unique_cmdb_count       overall_earliest_dt         overall_latest_dt  total_rows_all_files
0  [IG01, IG02, MG01, MG02, Mysql01, Mysql02, Red...                 29 2021-03-25 18:00:00+08:00 2021-03-25 18:30:00+08:00                 37070)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz
from pathlib import Path

tz = pytz.timezone('Asia/Shanghai')

# Reuse metric_df if present in kernel; otherwise load from CSV
if 'metric_df' not in globals():
    metric_path = Path("metric.csv")
    if metric_path.exists():
        metric_df = pd.read_csv(metric_path)
    else:
        metric_df = pd.DataFrame(columns=["timestamp","cmdb_id","kpi_name","value"])
# Ensure dt column exists and is timezone-aware Asia/Shanghai
if 'dt' not in metric_df.columns or metric_df['dt'].isna().all():
    if 'timestamp' in metric_df.columns and not metric_df.empty:
        metric_df['dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        metric_df['dt'] = pd.Series(dtype='datetime64[ns]')

# 1) Compute global thresholds (p95 and p5) for each (cmdb_id, kpi_name)
if not metric_df.empty:
    metric_thresholds = (
        metric_df
        .groupby(['cmdb_id', 'kpi_name'])['value']
        .agg(p95=lambda s: float(s.quantile(0.95)), p5=lambda s: float(s.quantile(0.05)))
        .reset_index()
    )
else:
    metric_thresholds = pd.DataFrame(columns=['cmdb_id','kpi_name','p95','p5'])

# 2) Filter metric.csv to the incident time range (inclusive)
start = pd.to_datetime('2021-03-25 18:00:00').tz_localize(tz)
end = pd.to_datetime('2021-03-25 18:30:00').tz_localize(tz)
if not metric_df.empty:
    metric_window = metric_df[(metric_df['dt'] >= start) & (metric_df['dt'] <= end)].copy()
else:
    metric_window = metric_df.copy()

# 3) Compute anomaly summary for groups present in metric_window
if metric_window.empty:
    # empty results
    anomaly_summary = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','p95','p5','high_count','low_count',
        'earliest_high_ts','earliest_low_ts','max_value','min_value',
        'severity_high','severity_low'
    ])
else:
    # merge thresholds into the window (thresholds computed globally before filtering)
    metric_w_t = metric_window.merge(metric_thresholds, on=['cmdb_id','kpi_name'], how='left')

    # booleans for anomalies
    metric_w_t['is_high'] = metric_w_t['value'] >= metric_w_t['p95']
    metric_w_t['is_low'] = metric_w_t['value'] <= metric_w_t['p5']

    # counts and max/min per group
    counts = metric_w_t.groupby(['cmdb_id','kpi_name'])[['is_high','is_low']].sum().astype(int).rename(columns={'is_high':'high_count','is_low':'low_count'}).reset_index()
    extreme = metric_w_t.groupby(['cmdb_id','kpi_name'])['value'].agg(max_value='max', min_value='min').reset_index()

    # earliest timestamps for high/low (may have missing groups)
    earliest_high = (
        metric_w_t[metric_w_t['is_high']]
        .groupby(['cmdb_id','kpi_name'])['dt']
        .min()
        .reset_index()
        .rename(columns={'dt':'earliest_high_ts'})
    )
    earliest_low = (
        metric_w_t[metric_w_t['is_low']]
        .groupby(['cmdb_id','kpi_name'])['dt']
        .min()
        .reset_index()
        .rename(columns={'dt':'earliest_low_ts'})
    )

    # base summary: unique groups in the window
    groups_in_window = metric_w_t[['cmdb_id','kpi_name']].drop_duplicates()

    anomaly_summary = (
        groups_in_window
        .merge(metric_thresholds, on=['cmdb_id','kpi_name'], how='left')
        .merge(counts, on=['cmdb_id','kpi_name'], how='left')
        .merge(extreme, on=['cmdb_id','kpi_name'], how='left')
        .merge(earliest_high, on=['cmdb_id','kpi_name'], how='left')
        .merge(earliest_low, on=['cmdb_id','kpi_name'], how='left')
    )

    # fill NaNs for counts with 0
    anomaly_summary['high_count'] = anomaly_summary['high_count'].fillna(0).astype(int)
    anomaly_summary['low_count'] = anomaly_summary['low_count'].fillna(0).astype(int)

    # compute severity metrics safely
    def compute_severity_high(row):
        if row['high_count'] > 0 and pd.notna(row['p95']) and row['p95'] != 0:
            return float((row['max_value'] - row['p95']) / row['p95'])
        return np.nan

    def compute_severity_low(row):
        if row['low_count'] > 0 and pd.notna(row['p5']) and row['p5'] != 0:
            return float((row['p5'] - row['min_value']) / row['p5'])
        return np.nan

    anomaly_summary['severity_high'] = anomaly_summary.apply(compute_severity_high, axis=1)
    anomaly_summary['severity_low'] = anomaly_summary.apply(compute_severity_low, axis=1)

    # Ensure timestamp columns are timezone-aware or NaT
    anomaly_summary['earliest_high_ts'] = pd.to_datetime(anomaly_summary['earliest_high_ts'])
    anomaly_summary['earliest_low_ts'] = pd.to_datetime(anomaly_summary['earliest_low_ts'])

# 4) Return top 20 rows sorted by max(high_count, low_count) desc and anomaly count
if anomaly_summary.empty:
    anomaly_summary_top20 = anomaly_summary.head(20)
    anomaly_count = 0
else:
    anomaly_summary['max_anom_count'] = anomaly_summary[['high_count','low_count']].max(axis=1)
    anomaly_summary_sorted = anomaly_summary.sort_values(['max_anom_count','high_count','low_count'], ascending=[False,False,False])
    cols = ['cmdb_id','kpi_name','p95','p5','high_count','low_count','earliest_high_ts','earliest_low_ts','max_value','min_value','severity_high','severity_low']
    anomaly_summary_top20 = anomaly_summary_sorted[cols].head(20)
    anomaly_count = int(((anomaly_summary['high_count'] > 0) | (anomaly_summary['low_count'] > 0)).sum())

# Keep metric_thresholds and metric_window in kernel; display compact outputs
anomaly_summary_top20, anomaly_count

```
Out[2]:
```
```python
# Compose a concise plain-English summary based on anomaly_summary_top20 and anomaly_count in the kernel.
# Display as a single string variable (compact).

# Safely access variables from kernel
ams_top = anomaly_summary_top20 if 'anomaly_summary_top20' in globals() else None
ams_count = anomaly_count if 'anomaly_count' in globals() else None

# Build summary
lines = []
lines.append("Anomaly summary for metrics in window 2021-03-25 18:00:00+08:00 to 2021-03-25 18:30:00+08:00:")
lines.append(f"- Number of (cmdb_id, kpi_name) groups with any anomaly (high or low): {ams_count}")
lines.append("- Observations:")
lines.append("  * Many groups show both high_count and low_count equal to the full-window point count (31),")
lines.append("    often because p95==p5 and max==min for those series, yielding severity 0 or NaN.")
lines.append("  * When severity is defined it is computed as relative difference to the threshold (p95 or p5).")

# Add top example rows (up to 3) for quick inspection
if ams_top is not None and not ams_top.empty:
    examples = []
    display_cols = ['cmdb_id','kpi_name','p95','p5','high_count','low_count','max_value','min_value','severity_high','severity_low']
    for _, r in ams_top.head(3).iterrows():
        examples.append(
            f"{r['cmdb_id']} | {r['kpi_name']} | high_count={int(r['high_count'])} | low_count={int(r['low_count'])} | "
            f"max={r['max_value']} | min={r['min_value']} | sev_high={r['severity_high']} | sev_low={r['severity_low']}"
        )
    lines.append("- Top example groups (3):")
    lines += ["    " + e for e in examples]
else:
    lines.append("- No anomaly summary rows available to show.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(          cmdb_id                                           kpi_name           p95            p5  high_count  low_count          earliest_high_ts           earliest_low_ts     max_value     min_value  severity_high  severity_low
0            IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09  2.040136e+09          31         31 2021-03-25 18:00:00+08:00 2021-03-25 18:00:00+08:00  2.040136e+09  2.040136e+09            0.0           0.0
3            IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.846119e+07  5.846119e+07          31         31 2021-03-25 18:00:00+08:00 2021-03-25 18:00:00+08:00  5.846119e+07  5.846119e+07            0.0           0.0
22           IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09  2.040136e+09          31         31 2021-03-25 18:00:00+08:00 2021-03-25 18:00:00+08:00  2.040136e+09  2.040136e+09            0.0           0.0
85           MG01           JVM-Memory_7779_JVM_Memory_HeapMemoryMax  2.040136e+09  2.040136e+09          31         31 2021-03-25 18:00:00+08:00 2021-03-25 18:00:00+08:00  2.040136e+09  2.040136e+09            0.0           0.0
88           MG01        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed  5.886139e+07  5.886139e+07          31         31 2021-03-25 18:00:00+08:00 2021-03-25 18:00:00+08:00  5.886139e+07  5.886139e+07            0.0           0.0
122          MG01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...  3.000000e-01  3.000000e-01          31         31 2021-03-25 18:00:00+08:00 2021-03-25 18:00:00+08:00  3.000000e-01  3.000000e-01            0.0           0.0
128          MG02           JVM-Memory_7779_JVM_Memory_HeapMemoryMax  2.040136e+09  2.040136e+09          31         31 2021-03-25 18:00:00+08:00 2021-03-25 18:00:00+08:00  2.040136e+09  2.040136e+09            0.0           0.0
131          MG02        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed  5.902190e+07  5.902190e+07          31         31 2021-03-25 18:00:00+08:00 2021-03-25 18:00:00+08:00  5.902190e+07  5.902190e+07            0.0           0.0
209          MG02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies  1.000000e+00  1.000000e+00          31         31 2021-03-25 18:00:00+08:00 2021-03-25 18:00:00+08:00  1.000000e+00  1.000000e+00            0.0           0.0
212          MG02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...  3.000000e-01  3.000000e-01          31         31 2021-03-25 18:00:00+08:00 2021-03-25 18:00:00+08:00  3.000000e-01  3.000000e-01            0.0           0.0
213          MG02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount  3.000000e+00  3.000000e+00          31         31 2021-03-25 18:00:00+08:00 2021-03-25 18:00:00+08:00  3.000000e+00  3.000000e+00            0.0           0.0
218       Mysql01         Mysql-MySQL_3306_GetConnectedStateOfMysqld  1.000000e+00  1.000000e+00          31         31 2021-03-25 18:00:00+08:00 2021-03-25 18:00:00+08:00  1.000000e+00  1.000000e+00            0.0           0.0
219       Mysql01           Mysql-MySQL_3306_GetResponseTimeOfMysqld  0.000000e+00  0.000000e+00          31         31 2021-03-25 18:00:00+08:00 2021-03-25 18:00:00+08:00  0.000000e+00  0.000000e+00            NaN           NaN
291       Mysql02         Mysql-MySQL_3306_GetConnectedStateOfMysqld  1.000000e+00  1.000000e+00          31         31 2021-03-25 18:00:00+08:00 2021-03-25 18:00:00+08:00  1.000000e+00  1.000000e+00            0.0           0.0
292       Mysql02           Mysql-MySQL_3306_GetResponseTimeOfMysqld  0.000000e+00  0.000000e+00          31         31 2021-03-25 18:00:00+08:00 2021-03-25 18:00:00+08:00  0.000000e+00  0.000000e+00            NaN           NaN
432       Redis02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT  0.000000e+00  0.000000e+00          31         31 2021-03-25 18:00:00+08:00 2021-03-25 18:00:00+08:00  0.000000e+00  0.000000e+00            NaN           NaN
433       Redis02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT  0.000000e+00  0.000000e+00          31         31 2021-03-25 18:00:00+08:00 2021-03-25 18:00:00+08:00  0.000000e+00  0.000000e+00            NaN           NaN
435       Redis02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies  0.000000e+00  0.000000e+00          31         31 2021-03-25 18:00:00+08:00 2021-03-25 18:00:00+08:00  0.000000e+00  0.000000e+00            NaN           NaN
438       Redis02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount  4.000000e+00  4.000000e+00          31         31 2021-03-25 18:00:00+08:00 2021-03-25 18:00:00+08:00  4.000000e+00  4.000000e+00            0.0           0.0
475  ServiceTest1                                             app.rr  1.000000e+02  1.000000e+02          31         31 2021-03-25 18:00:00+08:00 2021-03-25 18:00:00+08:00  1.000000e+02  1.000000e+02            0.0           0.0, 1759)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse trace_df from kernel if available, otherwise load
if 'trace_df' not in globals():
    trace_df = pd.read_csv("trace.csv")

# Ensure dt column exists and is timezone-aware Asia/Shanghai
if 'dt' not in trace_df.columns or trace_df['dt'].isna().all():
    if 'timestamp' in trace_df.columns and not trace_df.empty:
        trace_df['dt'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        trace_df['dt'] = pd.Series(dtype='datetime64[ns]')

# 1) Global thresholds (p95 and p5) per (cmdb_id, trace_name) computed from entire series
if not trace_df.empty:
    trace_thresholds = (
        trace_df
        .groupby(['cmdb_id', 'trace_name'])['value']
        .agg(p95=lambda s: float(s.quantile(0.95)), p5=lambda s: float(s.quantile(0.05)))
        .reset_index()
    )
else:
    trace_thresholds = pd.DataFrame(columns=['cmdb_id','trace_name','p95','p5'])

# 2) Filter trace.csv to the incident window (inclusive)
start = pd.to_datetime('2021-03-25 18:00:00').tz_localize(tz)
end = pd.to_datetime('2021-03-25 18:30:00').tz_localize(tz)
if not trace_df.empty:
    trace_window = trace_df[(trace_df['dt'] >= start) & (trace_df['dt'] <= end)].copy()
else:
    trace_window = trace_df.copy()

# 3) Compute anomaly summary for groups present in trace_window
cols_out = [
    'cmdb_id','trace_name','p95','p5','high_count','low_count',
    'earliest_high_ts','earliest_low_ts','max_value','min_value','severity_high','severity_low'
]

if trace_window.empty:
    trace_anomaly_summary = pd.DataFrame(columns=cols_out)
else:
    # merge global thresholds into the window (thresholds computed before filtering)
    tw = trace_window.merge(trace_thresholds, on=['cmdb_id','trace_name'], how='left')

    # mark anomalies
    tw['is_high'] = tw['value'] >= tw['p95']
    tw['is_low'] = tw['value'] <= tw['p5']

    # counts
    counts = tw.groupby(['cmdb_id','trace_name'])[['is_high','is_low']].sum().rename(columns={'is_high':'high_count','is_low':'low_count'}).reset_index()
    counts['high_count'] = counts['high_count'].astype(int)
    counts['low_count'] = counts['low_count'].astype(int)

    # extremes
    extremes = tw.groupby(['cmdb_id','trace_name'])['value'].agg(max_value='max', min_value='min').reset_index()

    # earliest timestamps
    e_high = tw[tw['is_high']].groupby(['cmdb_id','trace_name'])['dt'].min().reset_index().rename(columns={'dt':'earliest_high_ts'})
    e_low = tw[tw['is_low']].groupby(['cmdb_id','trace_name'])['dt'].min().reset_index().rename(columns={'dt':'earliest_low_ts'})

    # base groups in window
    groups = tw[['cmdb_id','trace_name']].drop_duplicates()

    trace_anomaly_summary = (
        groups
        .merge(trace_thresholds, on=['cmdb_id','trace_name'], how='left')
        .merge(counts, on=['cmdb_id','trace_name'], how='left')
        .merge(extremes, on=['cmdb_id','trace_name'], how='left')
        .merge(e_high, on=['cmdb_id','trace_name'], how='left')
        .merge(e_low, on=['cmdb_id','trace_name'], how='left')
    )

    # fill counts with 0
    trace_anomaly_summary['high_count'] = trace_anomaly_summary['high_count'].fillna(0).astype(int)
    trace_anomaly_summary['low_count'] = trace_anomaly_summary['low_count'].fillna(0).astype(int)

    # compute severities
    def sev_high(row):
        if row['high_count'] > 0 and pd.notna(row['p95']) and row['p95'] != 0:
            return float((row['max_value'] - row['p95']) / row['p95'])
        return np.nan

    def sev_low(row):
        if row['low_count'] > 0 and pd.notna(row['p5']) and row['p5'] != 0:
            return float((row['p5'] - row['min_value']) / row['p5'])
        return np.nan

    trace_anomaly_summary['severity_high'] = trace_anomaly_summary.apply(sev_high, axis=1)
    trace_anomaly_summary['severity_low'] = trace_anomaly_summary.apply(sev_low, axis=1)

    # ensure timestamp columns are proper datetimes (tz-aware or NaT)
    trace_anomaly_summary['earliest_high_ts'] = pd.to_datetime(trace_anomaly_summary['earliest_high_ts'])
    trace_anomaly_summary['earliest_low_ts'] = pd.to_datetime(trace_anomaly_summary['earliest_low_ts'])

# 4) Prepare top 20 sorted by max(high_count, low_count) desc, then by max(severity_high,severity_low) desc
if trace_anomaly_summary.empty:
    trace_anomaly_summary_top20 = trace_anomaly_summary.head(20)
    trace_anomaly_count = 0
else:
    trace_anomaly_summary['max_anom_count'] = trace_anomaly_summary[['high_count','low_count']].max(axis=1)
    # compute max severity for sorting (NaN -> -1 to push to bottom)
    trace_anomaly_summary['max_severity'] = trace_anomaly_summary[['severity_high','severity_low']].max(axis=1)
    trace_anomaly_summary['max_severity_sort'] = trace_anomaly_summary['max_severity'].fillna(-1.0)

    trace_anomaly_summary_sorted = trace_anomaly_summary.sort_values(
        by=['max_anom_count','max_severity_sort'],
        ascending=[False, False]
    )

    trace_anomaly_summary_top20 = trace_anomaly_summary_sorted[
        ['cmdb_id','trace_name','p95','p5','high_count','low_count','earliest_high_ts','earliest_low_ts','max_value','min_value','severity_high','severity_low']
    ].head(20)

    trace_anomaly_count = int(((trace_anomaly_summary['high_count'] > 0) | (trace_anomaly_summary['low_count'] > 0)).sum())

# Keep trace_thresholds and trace_window in kernel; display compact outputs
trace_anomaly_summary_top20, trace_anomaly_count

```
Out[3]:
```
```python
# Build a concise plain-English summary from trace_anomaly_summary_top20 and trace_anomaly_count in the kernel.
# This will produce a single compact string as the output.

def safe_head_rows(df, n=3):
    if df is None or df.empty:
        return []
    return df.head(n).to_dict(orient='records')

top20 = trace_anomaly_summary_top20 if 'trace_anomaly_summary_top20' in globals() else None
count = trace_anomaly_count if 'trace_anomaly_count' in globals() else 0

lines = []
lines.append("Trace anomaly summary for window 2021-03-25 18:00:00+08:00 to 2021-03-25 18:30:00+08:00:")
lines.append(f"- Number of (cmdb_id, trace_name) groups with any anomaly: {count}")
lines.append("- Top groups (sorted by anomaly count and severity). Examples:")

examples = safe_head_rows(top20, 3)
if examples:
    for e in examples:
        lines.append(
            f"  * {e['cmdb_id']} | {e['trace_name']} — high_count={int(e['high_count'])}, low_count={int(e['low_count'])}, "
            f"max={e['max_value']}, min={e['min_value']}, sev_high={e['severity_high']}, sev_low={e['severity_low']}"
        )
else:
    lines.append("  * No anomaly rows to show.")

lines.append("- Notes:")
lines.append("  * p95/p5 thresholds were computed globally from the full trace series before filtering the window.")
lines.append("  * severity_high = (max_value - p95)/p95; severity_low = (p5 - min_value)/p5 (NaN if undefined).")
lines.append("  * Many groups show multiple high and low occurrences; highest example counts include docker and MG components.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id                        trace_name         p95       p5  high_count  low_count          earliest_high_ts           earliest_low_ts  max_value  min_value  severity_high  severity_low
217  dockerB1           trace.self.duration_p95    0.011000   0.0090           9          3 2021-03-25 18:00:00+08:00 2021-03-25 18:10:00+08:00     0.0138    0.00315       0.254545      0.650000
184  dockerA1           trace.self.duration_p95    0.009000   0.0010           6          7 2021-03-25 18:04:00+08:00 2021-03-25 18:15:00+08:00     1.0090    0.00100     111.111111      0.000000
199  dockerA2           trace.self.duration_p95    0.009000   0.0010           5          7 2021-03-25 18:05:00+08:00 2021-03-25 18:00:00+08:00     0.2088    0.00100      22.200000      0.000000
118  Tomcat01           trace.self.duration_p95    0.068400   0.0240           2          7 2021-03-25 18:05:00+08:00 2021-03-25 18:01:00+08:00     0.1062    0.01190       0.552632      0.504167
235  dockerB2           trace.self.duration_p95    0.048800   0.0090           2          4 2021-03-25 18:05:00+08:00 2021-03-25 18:11:00+08:00     0.1874    0.00000       2.840164      1.000000
83       MG02     trace.from_dockerA1.row_count   51.650000  13.0000           2          4 2021-03-25 18:05:00+08:00 2021-03-25 18:15:00+08:00    56.0000   10.00000       0.084221      0.230769
191  dockerA1           trace.to_MG02.row_count   51.650000  13.0000           2          4 2021-03-25 18:05:00+08:00 2021-03-25 18:15:00+08:00    56.0000   10.00000       0.084221      0.230769
154  Tomcat03           trace.self.duration_p95    0.328525   0.0240           2          3 2021-03-25 18:05:00+08:00 2021-03-25 18:02:00+08:00     0.8140    0.02300       1.477741      0.041667
98       MG02       trace.to_dockerA1.row_count  146.000000  59.5000           3          2 2021-03-25 18:03:00+08:00 2021-03-25 18:29:00+08:00   161.0000    1.00000       0.102740      0.983193
182  dockerA1         trace.from_MG02.row_count  146.000000  59.5000           3          2 2021-03-25 18:03:00+08:00 2021-03-25 18:29:00+08:00   161.0000    1.00000       0.102740      0.983193
68       MG01       trace.to_dockerB2.row_count   69.500000  45.0000           2          3 2021-03-25 18:14:00+08:00 2021-03-25 18:03:00+08:00    71.0000    1.00000       0.021583      0.977778
227  dockerB2         trace.from_MG01.row_count   69.500000  45.0000           2          3 2021-03-25 18:14:00+08:00 2021-03-25 18:03:00+08:00    71.0000    1.00000       0.021583      0.977778
29       IG02       trace.to_Tomcat04.row_count  162.000000  95.5000           3          2 2021-03-25 18:00:00+08:00 2021-03-25 18:06:00+08:00   172.0000    3.00000       0.061728      0.968586
167  Tomcat04         trace.from_IG02.row_count  162.000000  95.5000           3          2 2021-03-25 18:00:00+08:00 2021-03-25 18:06:00+08:00   172.0000    3.00000       0.061728      0.968586
52       MG01  trace.from_dockerB2.duration_p95    0.949660   0.0090           2          3 2021-03-25 18:01:00+08:00 2021-03-25 18:06:00+08:00     1.0183    0.00700       0.072278      0.222222
238  dockerB2        trace.to_MG01.duration_p95    0.949660   0.0090           2          3 2021-03-25 18:01:00+08:00 2021-03-25 18:06:00+08:00     1.0183    0.00700       0.072278      0.222222
82       MG02  trace.from_dockerA1.duration_p95    0.870170   0.0098           2          3 2021-03-25 18:01:00+08:00 2021-03-25 18:19:00+08:00     1.0174    0.00810       0.169197      0.173469
190  dockerA1        trace.to_MG02.duration_p95    0.870170   0.0098           2          3 2021-03-25 18:01:00+08:00 2021-03-25 18:19:00+08:00     1.0174    0.00810       0.169197      0.173469
107      MG02       trace.to_dockerB2.row_count   72.000000  43.9000           3          2 2021-03-25 18:00:00+08:00 2021-03-25 18:12:00+08:00    73.0000   38.00000       0.013889      0.134396
230  dockerB2         trace.from_MG02.row_count   72.000000  43.9000           3          2 2021-03-25 18:00:00+08:00 2021-03-25 18:12:00+08:00    73.0000   38.00000       0.013889      0.134396, 243)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing trace anomaly summary if available
if 'trace_anomaly_summary' in globals():
    tas = trace_anomaly_summary.copy()
elif 'trace_anomaly_summary_top20' in globals():
    tas = trace_anomaly_summary_top20.copy()
else:
    tas = pd.DataFrame(columns=[
        'cmdb_id','trace_name','p95','p5','high_count','low_count',
        'earliest_high_ts','earliest_low_ts','max_value','min_value','severity_high','severity_low'
    ])

# Ensure columns exist
for c in ['high_count','low_count','severity_high','severity_low','earliest_high_ts','earliest_low_ts']:
    if c not in tas.columns:
        tas[c] = np.nan

# Normalize timestamp columns to timezone-aware (Asia/Shanghai) where possible
def ensure_tz(series):
    if series.dtype == 'O':
        # try convert to datetime
        s = pd.to_datetime(series, errors='coerce')
    else:
        s = pd.to_datetime(series, errors='coerce')
    # If tz-naive, localize; if tz-aware, convert
    if not s.empty:
        if s.dt.tz is None:
            try:
                s = s.dt.tz_localize(tz)
            except Exception:
                # if already tz-aware or cannot localize, attempt convert (safe)
                s = s.dt.tz_convert(tz)
    return s

tas['earliest_high_ts'] = ensure_tz(tas.get('earliest_high_ts', pd.Series(dtype='datetime64[ns]')))
tas['earliest_low_ts'] = ensure_tz(tas.get('earliest_low_ts', pd.Series(dtype='datetime64[ns]')))

# Aggregations per cmdb_id
grouped = tas.groupby('cmdb_id')

per_cmdb = grouped.agg(
    num_trace_groups = ('trace_name', 'nunique'),
    num_groups_with_high = ('high_count', lambda s: int((s > 0).sum())),
    num_groups_with_low = ('low_count', lambda s: int((s > 0).sum())),
    total_high_count = ('high_count', 'sum'),
    total_low_count = ('low_count', 'sum'),
    earliest_high_ts = ('earliest_high_ts', lambda s: s.min() if s.notna().any() else pd.NaT),
    earliest_low_ts = ('earliest_low_ts', lambda s: s.min() if s.notna().any() else pd.NaT),
    max_severity_high = ('severity_high', lambda s: float(s.dropna().max()) if s.dropna().size>0 else np.nan),
    max_severity_low = ('severity_low', lambda s: float(s.dropna().max()) if s.dropna().size>0 else np.nan)
).reset_index()

# number_of_trace_groups_with_any_anomaly
# Need to compute per-cmdb count of groups where high_count>0 or low_count>0
anom_flag = tas.assign(any_anom = ((tas['high_count']>0) | (tas['low_count']>0)).astype(int))
any_counts = anom_flag.groupby('cmdb_id')['any_anom'].sum().reset_index().rename(columns={'any_anom':'num_groups_with_any_anomaly'})

per_cmdb = per_cmdb.merge(any_counts, on='cmdb_id', how='left')
per_cmdb['num_groups_with_any_anomaly'] = per_cmdb['num_groups_with_any_anomaly'].fillna(0).astype(int)

# Prepare example trace rows per cmdb_id (up to 3) prioritized by severity_high desc, then high_count desc
def make_examples(df):
    # sort by severity_high (NaN last) then high_count desc
    df_sorted = df.copy()
    df_sorted['sev_for_sort'] = df_sorted['severity_high'].fillna(-np.inf)
    df_sorted = df_sorted.sort_values(by=['sev_for_sort','high_count'], ascending=[False, False])
    examples = []
    for _, r in df_sorted.head(3).iterrows():
        eht = r['earliest_high_ts']
        eht_str = eht.isoformat() if pd.notna(eht) else ''
        sev = r['severity_high']
        sev_str = (f"{sev:.6g}" if pd.notna(sev) else '')
        examples.append(f"{r['trace_name']}|hc={int(r['high_count'])}|eht={eht_str}|sev={sev_str}")
    return "; ".join(examples)

examples_series = tas.groupby('cmdb_id').apply(make_examples).reset_index().rename(columns={0:'top_examples'})

per_cmdb = per_cmdb.merge(examples_series, on='cmdb_id', how='left')
per_cmdb['top_examples'] = per_cmdb['top_examples'].fillna('')

# Count how many cmdb_id had any anomaly (num_groups_with_high>0 or num_groups_with_low>0)
components_with_any_anomaly_count = int(((per_cmdb['num_groups_with_high']>0) | (per_cmdb['num_groups_with_low']>0)).sum())

# Sort top 20 rows by (num_groups_with_any_anomaly desc, total_high_count desc)
per_cmdb_sorted = per_cmdb.sort_values(by=['num_groups_with_any_anomaly','total_high_count'], ascending=[False, False])
per_cmdb_summary_top20 = per_cmdb_sorted.head(20)[[
    'cmdb_id','num_trace_groups','num_groups_with_high','num_groups_with_low',
    'num_groups_with_any_anomaly','total_high_count','total_low_count',
    'earliest_high_ts','earliest_low_ts','max_severity_high','max_severity_low','top_examples'
]]

# Final compact outputs (stored in kernel variables)
per_cmdb_summary_top20, components_with_any_anomaly_count

```
Out[4]:
```
```python
# Build a concise plain-English summary based on per_cmdb_summary_top20 and components_with_any_anomaly_count in the kernel.
pc = per_cmdb_summary_top20 if 'per_cmdb_summary_top20' in globals() else None
count = components_with_any_anomaly_count if 'components_with_any_anomaly_count' in globals() else 0

lines = []
lines.append("Per-component trace-anomaly summary (window 2021-03-25 18:00:00+08:00 to 18:30:00+08:00):")
lines.append(f"- Components with any anomaly: {count}")
lines.append("- Top affected components (by number of trace groups with anomalies / total high_count / total_low_count):")

if pc is None or pc.empty:
    lines.append("  * No data available.")
else:
    # show top 6 compact examples
    top_n = pc.head(6)
    for _, r in top_n.iterrows():
        lines.append(
            f"  * {r['cmdb_id']}: groups={int(r['num_trace_groups'])}, "
            f"groups_with_any_anom={int(r['num_groups_with_any_anomaly'])}, "
            f"total_high={int(r['total_high_count'])}, total_low={int(r['total_low_count'])}, "
            f"earliest_high={r['earliest_high_ts']}, max_sev_high={r['max_severity_high']:.6g}, "
            f"examples=[{r['top_examples']}]"
        )

lines.append("- High-level observation: anomalies are widespread across MG, docker, and Tomcat components;")
lines.append("  several trace groups show multiple high and low occurrences within the 30-minute window.")
summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id  num_trace_groups  num_groups_with_high  num_groups_with_low  num_groups_with_any_anomaly  total_high_count  total_low_count          earliest_high_ts           earliest_low_ts  max_severity_high  max_severity_low                                       top_examples
3       MG02                39                    39                   39                           39                80               81 2021-03-25 18:00:00+08:00 2021-03-25 18:01:00+08:00           5.597203          0.989950  trace.to_dockerA1.duration_mean|hc=2|eht=2021-...
2       MG01                39                    39                   39                           39                78               81 2021-03-25 18:00:00+08:00 2021-03-25 18:01:00+08:00           1.955180          0.990204  trace.to_dockerA2.duration_mean|hc=2|eht=2021-...
10  dockerB1                18                    18                   18                           18                40               34 2021-03-25 18:00:00+08:00 2021-03-25 18:00:00+08:00           1.429977          0.991803  trace.to_MG02.duration_p95|hc=2|eht=2021-03-25...
11  dockerB2                18                    18                   18                           18                34               37 2021-03-25 18:00:00+08:00 2021-03-25 18:00:00+08:00           2.840164          1.000000  trace.self.duration_p95|hc=2|eht=2021-03-25T18...
4   Tomcat01                18                    18                   18                           18                33               38 2021-03-25 18:00:00+08:00 2021-03-25 18:00:00+08:00           0.552632          0.989950  trace.self.duration_p95|hc=2|eht=2021-03-25T18...
5   Tomcat02                18                    18                   18                           18                33               33 2021-03-25 18:00:00+08:00 2021-03-25 18:00:00+08:00           3.203881          0.989247  trace.self.duration_p95|hc=2|eht=2021-03-25T18...
6   Tomcat03                18                    18                   18                           18                33               34 2021-03-25 18:00:00+08:00 2021-03-25 18:00:00+08:00           1.477741          0.979695  trace.self.duration_p95|hc=2|eht=2021-03-25T18...
8   dockerA1                15                    15                   15                           15                35               39 2021-03-25 18:01:00+08:00 2021-03-25 18:06:00+08:00         111.111111          0.998970  trace.self.duration_p95|hc=6|eht=2021-03-25T18...
9   dockerA2                15                    15                   15                           15                33               35 2021-03-25 18:01:00+08:00 2021-03-25 18:00:00+08:00          22.200000          0.982688  trace.self.duration_p95|hc=5|eht=2021-03-25T18...
1       IG02                15                    15                   15                           15                31               30 2021-03-25 18:00:00+08:00 2021-03-25 18:02:00+08:00           0.272593          0.989247  trace.to_Tomcat02.duration_mean|hc=2|eht=2021-...
7   Tomcat04                15                    15                   15                           15                31               30 2021-03-25 18:00:00+08:00 2021-03-25 18:01:00+08:00           0.224292          0.989848  trace.self.duration_mean|hc=2|eht=2021-03-25T1...
0       IG01                15                    15                   15                           15                30               30 2021-03-25 18:01:00+08:00 2021-03-25 18:01:00+08:00           0.286695          0.989848  trace.to_Tomcat03.duration_mean|hc=2|eht=2021-..., 12)```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse trace_anomaly_summary if available, otherwise fallback to top20
if 'trace_anomaly_summary' in globals():
    tas = trace_anomaly_summary.copy()
elif 'trace_anomaly_summary_top20' in globals():
    tas = trace_anomaly_summary_top20.copy()
else:
    tas = pd.DataFrame(columns=[
        'cmdb_id','trace_name','p95','p5','high_count','low_count',
        'earliest_high_ts','earliest_low_ts','max_value','min_value','severity_high'
    ])

# Ensure timestamp columns are datetimes and timezone-aware (Asia/Shanghai)
for col in ['earliest_high_ts','earliest_low_ts']:
    if col in tas.columns:
        tas[col] = pd.to_datetime(tas[col], errors='coerce')
        # localize naive datetimes to Asia/Shanghai
        if not tas.empty:
            if tas[col].dt.tz is None:
                try:
                    tas[col] = tas[col].dt.tz_localize(tz)
                except Exception:
                    # if already tz-aware or localization failed, attempt convert
                    tas[col] = tas[col].dt.tz_convert(tz)

# Filter to MG01 or MG02 and high_count > 0
filtered = tas.loc[
    tas['cmdb_id'].isin(['MG01','MG02']) & (tas['high_count'] > 0),
    ['cmdb_id','trace_name','p95','p5','high_count','low_count','earliest_high_ts','earliest_low_ts','max_value','min_value','severity_high']
].copy()

# Compute the minimum (earliest) earliest_high_ts among these filtered rows
if not filtered.empty and filtered['earliest_high_ts'].notna().any():
    min_earliest_high_ts = filtered['earliest_high_ts'].min()
else:
    min_earliest_high_ts = pd.NaT

# Keep full filtered table in kernel variable `mg_filtered_trace_anomalies`
mg_filtered_trace_anomalies = filtered

# Display compact outputs: up to 50 rows and the single earliest timestamp
mg_filtered_trace_anomalies.head(50), min_earliest_high_ts

```
Out[5]:
```
```python
# Compose a concise plain-English summary from the filtered MG trace anomalies in the kernel.
# Uses variables: mg_filtered_trace_anomalies, min_earliest_high_ts

import pandas as pd

mg = mg_filtered_trace_anomalies if 'mg_filtered_trace_anomalies' in globals() else pd.DataFrame()
min_ts = min_earliest_high_ts if 'min_earliest_high_ts' in globals() else pd.NaT

# Safely compute counts and top examples
row_count = int(mg.shape[0]) if not mg.empty else 0
by_cmdb = mg.groupby('cmdb_id').size().to_dict() if not mg.empty else {}

# Top 3 examples by severity_high then high_count
if not mg.empty:
    ex = mg.sort_values(['severity_high','high_count'], ascending=[False, False]).head(3)
    examples = []
    for _, r in ex.iterrows():
        eht = r['earliest_high_ts']
        eht_str = eht.isoformat() if pd.notna(eht) else ''
        sev = r['severity_high']
        sev_str = f"{sev:.6g}" if pd.notna(sev) else ''
        examples.append(f"{r['cmdb_id']} | {r['trace_name']} | hc={int(r['high_count'])} | eht={eht_str} | sev={sev_str}")
else:
    examples = []

summary_lines = [
    "Filtered MG trace-anomalies summary (cmdb_id in MG01 or MG02, high_count>0):",
    f"- Total filtered rows: {row_count}",
    f"- Rows by component: {by_cmdb}",
    f"- Earliest 'earliest_high_ts' among these rows: {min_ts}",
    "- Top example trace anomalies (up to 3):"
]
if examples:
    summary_lines += [f"  * {e}" for e in examples]
else:
    summary_lines.append("  * (no example rows)")

summary = "\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id                         trace_name         p95         p5  high_count  low_count          earliest_high_ts           earliest_low_ts   max_value  min_value  severity_high
30    MG01  trace.from_Tomcat01.duration_mean    0.311260   0.055391           2          2 2021-03-25 18:05:00+08:00 2021-03-25 18:09:00+08:00    0.360194   0.009000       0.157213
31    MG01   trace.from_Tomcat01.duration_p95    2.083300   0.091875           2          2 2021-03-25 18:05:00+08:00 2021-03-25 18:22:00+08:00    2.093450   0.017900       0.004872
32    MG01      trace.from_Tomcat01.row_count  171.000000  98.500000           2          2 2021-03-25 18:01:00+08:00 2021-03-25 18:20:00+08:00  187.000000   4.000000       0.093567
33    MG01  trace.from_Tomcat02.duration_mean    0.308079   0.056759           2          2 2021-03-25 18:05:00+08:00 2021-03-25 18:02:00+08:00    0.410000   0.014000       0.330827
34    MG01   trace.from_Tomcat02.duration_p95    2.080550   0.103350           2          2 2021-03-25 18:05:00+08:00 2021-03-25 18:02:00+08:00    2.566000   0.020300       0.233328
..     ...                                ...         ...        ...         ...        ...                       ...                       ...         ...        ...            ...
75    MG02  trace.from_Tomcat03.duration_mean    0.327863   0.063861           2          2 2021-03-25 18:27:00+08:00 2021-03-25 18:02:00+08:00    0.522000   0.060851       0.592126
76    MG02   trace.from_Tomcat03.duration_p95    2.069600   0.199400           2          2 2021-03-25 18:05:00+08:00 2021-03-25 18:11:00+08:00    2.095850   0.053550       0.012684
77    MG02      trace.from_Tomcat03.row_count  171.500000  92.500000           2          2 2021-03-25 18:16:00+08:00 2021-03-25 18:23:00+08:00  196.000000   2.000000       0.142857
78    MG02  trace.from_Tomcat04.duration_mean    0.238536   0.063736           2          2 2021-03-25 18:00:00+08:00 2021-03-25 18:01:00+08:00    0.272620   0.005000       0.142888
79    MG02   trace.from_Tomcat04.duration_p95    1.978525   0.067000           2          2 2021-03-25 18:00:00+08:00 2021-03-25 18:01:00+08:00    2.074000   0.005000       0.048256

[50 rows x 11 columns], Timestamp('2021-03-25 18:00:00+0800', tz='Asia/Shanghai'))

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse metric_df from kernel (loaded previously)
if 'metric_df' not in globals():
    metric_df = pd.read_csv("metric.csv")
# Ensure dt exists and is timezone-aware Asia/Shanghai
if 'dt' not in metric_df.columns or metric_df['dt'].isna().all():
    if 'timestamp' in metric_df.columns and not metric_df.empty:
        metric_df['dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        metric_df['dt'] = pd.Series(dtype='datetime64[ns]')

# Target components
targets = ['MG01', 'MG02']

# 1) Compute global thresholds (p95 and p5) for MG01 and MG02 BEFORE filtering the window
metric_mg_thresholds = (
    metric_df[metric_df['cmdb_id'].isin(targets)]
    .groupby(['cmdb_id','kpi_name'])['value']
    .agg(p95=lambda s: float(s.quantile(0.95)), p5=lambda s: float(s.quantile(0.05)))
    .reset_index()
)

# 2) Filter metric_df to the incident window and restrict to MG01/MG02
start = pd.to_datetime('2021-03-25 18:00:00').tz_localize(tz)
end = pd.to_datetime('2021-03-25 18:30:00').tz_localize(tz)
metric_mg_window = metric_df[
    (metric_df['cmdb_id'].isin(targets)) &
    (metric_df['dt'] >= start) &
    (metric_df['dt'] <= end)
].copy()

# 3) For each (cmdb_id,kpi_name) in the window compute anomaly summary, using thresholds computed globally
if metric_mg_window.empty:
    metric_mg_anomaly = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','p95','p5','high_count','low_count',
        'earliest_high_ts','earliest_low_ts','max_value','min_value','severity_high','severity_low'
    ])
else:
    # Merge global thresholds into window rows
    mm = metric_mg_window.merge(metric_mg_thresholds, on=['cmdb_id','kpi_name'], how='left')

    # Mark anomalies
    mm['is_high'] = mm['value'] >= mm['p95']
    mm['is_low'] = mm['value'] <= mm['p5']

    # Counts per group
    counts = mm.groupby(['cmdb_id','kpi_name'])[['is_high','is_low']].sum().rename(columns={'is_high':'high_count','is_low':'low_count'}).reset_index()
    counts['high_count'] = counts['high_count'].astype(int)
    counts['low_count'] = counts['low_count'].astype(int)

    # Max/min values in window
    extremes = mm.groupby(['cmdb_id','kpi_name'])['value'].agg(max_value='max', min_value='min').reset_index()

    # Earliest timestamps within window where anomalies occurred
    earliest_high = mm[mm['is_high']].groupby(['cmdb_id','kpi_name'])['dt'].min().reset_index().rename(columns={'dt':'earliest_high_ts'})
    earliest_low = mm[mm['is_low']].groupby(['cmdb_id','kpi_name'])['dt'].min().reset_index().rename(columns={'dt':'earliest_low_ts'})

    # Base groups present in window
    groups_in_window = mm[['cmdb_id','kpi_name']].drop_duplicates()

    metric_mg_anomaly = (
        groups_in_window
        .merge(metric_mg_thresholds, on=['cmdb_id','kpi_name'], how='left')
        .merge(counts, on=['cmdb_id','kpi_name'], how='left')
        .merge(extremes, on=['cmdb_id','kpi_name'], how='left')
        .merge(earliest_high, on=['cmdb_id','kpi_name'], how='left')
        .merge(earliest_low, on=['cmdb_id','kpi_name'], how='left')
    )

    # Fill missing counts with zero
    metric_mg_anomaly['high_count'] = metric_mg_anomaly['high_count'].fillna(0).astype(int)
    metric_mg_anomaly['low_count'] = metric_mg_anomaly['low_count'].fillna(0).astype(int)

    # Compute severity metrics
    def sev_high(r):
        if r['high_count'] > 0 and pd.notna(r['p95']) and r['p95'] != 0:
            return float((r['max_value'] - r['p95']) / r['p95'])
        return np.nan

    def sev_low(r):
        if r['low_count'] > 0 and pd.notna(r['p5']) and r['p5'] != 0:
            return float((r['p5'] - r['min_value']) / r['p5'])
        return np.nan

    metric_mg_anomaly['severity_high'] = metric_mg_anomaly.apply(sev_high, axis=1)
    metric_mg_anomaly['severity_low'] = metric_mg_anomaly.apply(sev_low, axis=1)

    # Ensure timestamp columns are timezone-aware datetimes or NaT
    metric_mg_anomaly['earliest_high_ts'] = pd.to_datetime(metric_mg_anomaly['earliest_high_ts'], errors='coerce')
    metric_mg_anomaly['earliest_low_ts'] = pd.to_datetime(metric_mg_anomaly['earliest_low_ts'], errors='coerce')
    # Localize naive timestamps to tz if needed
    for col in ['earliest_high_ts','earliest_low_ts']:
        if not metric_mg_anomaly.empty and metric_mg_anomaly[col].dt.tz is None:
            metric_mg_anomaly[col] = metric_mg_anomaly[col].dt.tz_localize(tz)

# 4) Prepare top 20 rows sorted by max(high_count, low_count) desc, then by max severity desc
if metric_mg_anomaly.empty:
    metric_mg_anomaly_top20 = metric_mg_anomaly.head(20)
    metric_mg_anomaly_count = 0
else:
    metric_mg_anomaly['max_anom_count'] = metric_mg_anomaly[['high_count','low_count']].max(axis=1)
    metric_mg_anomaly['max_severity'] = metric_mg_anomaly[['severity_high','severity_low']].max(axis=1).fillna(-np.inf)
    metric_mg_anomaly_sorted = metric_mg_anomaly.sort_values(by=['max_anom_count','max_severity'], ascending=[False, False])

    cols = ['cmdb_id','kpi_name','p95','p5','high_count','low_count','earliest_high_ts','earliest_low_ts','max_value','min_value','severity_high','severity_low']
    metric_mg_anomaly_top20 = metric_mg_anomaly_sorted[cols].head(20)
    metric_mg_anomaly_count = int(((metric_mg_anomaly['high_count'] > 0) | (metric_mg_anomaly['low_count'] > 0)).sum())

# Keep metric_mg_thresholds and metric_mg_window in kernel; display compact outputs
metric_mg_anomaly_top20, metric_mg_anomaly_count

```
Out[6]:
```
```python
# Compose a concise plain-English summary using kernel variables:
# metric_mg_anomaly_top20 and metric_mg_anomaly_count (computed previously).
mm_top = metric_mg_anomaly_top20 if 'metric_mg_anomaly_top20' in globals() else None
mm_count = metric_mg_anomaly_count if 'metric_mg_anomaly_count' in globals() else 0

lines = []
lines.append("Metric anomaly summary for MG01 and MG02 (window 2021-03-25 18:00:00+08:00 to 2021-03-25 18:30:00+08:00):")
lines.append(f"- Number of (cmdb_id, kpi_name) groups with any anomaly: {mm_count}")
lines.append("- Top anomalous metric groups (examples):")

if mm_top is None or mm_top.empty:
    lines.append("  * No metric anomaly groups found in the selected window.")
else:
    # show up to 3 example rows from the top20 table
    ex = mm_top.head(3)
    for _, r in ex.iterrows():
        eht = r['earliest_high_ts']
        elt = r['earliest_low_ts']
        eht_str = eht.isoformat() if pd.notna(eht) else ''
        elt_str = elt.isoformat() if pd.notna(elt) else ''
        sev_h = r['severity_high']
        sev_h_str = f"{sev_h:.6g}" if pd.notna(sev_h) else ''
        lines.append(
            f"  * {r['cmdb_id']} | {r['kpi_name']} — high_count={int(r['high_count'])}, low_count={int(r['low_count'])}, "
            f"max={r['max_value']}, min={r['min_value']}, earliest_high={eht_str}, earliest_low={elt_str}, sev_high={sev_h_str}"
        )

lines.append("- Notes:")
lines.append("  * Global p95/p5 thresholds were computed from the full metric series before window filtering.")
lines.append("  * Many groups show high_count and low_count equal to full-window counts (often p95==p5 and max==min), producing severity 0 or NaN.")
lines.append("  * Use the stored variables metric_mg_thresholds and metric_mg_window for deeper inspection if needed.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id                                           kpi_name           p95            p5  high_count  low_count          earliest_high_ts           earliest_low_ts     max_value     min_value  severity_high  severity_low
0      MG01           JVM-Memory_7779_JVM_Memory_HeapMemoryMax  2.040136e+09  2.040136e+09          31         31 2021-03-25 18:00:00+08:00 2021-03-25 18:00:00+08:00  2.040136e+09  2.040136e+09            0.0           0.0
3      MG01        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed  5.886139e+07  5.886139e+07          31         31 2021-03-25 18:00:00+08:00 2021-03-25 18:00:00+08:00  5.886139e+07  5.886139e+07            0.0           0.0
37     MG01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...  3.000000e-01  3.000000e-01          31         31 2021-03-25 18:00:00+08:00 2021-03-25 18:00:00+08:00  3.000000e-01  3.000000e-01            0.0           0.0
43     MG02           JVM-Memory_7779_JVM_Memory_HeapMemoryMax  2.040136e+09  2.040136e+09          31         31 2021-03-25 18:00:00+08:00 2021-03-25 18:00:00+08:00  2.040136e+09  2.040136e+09            0.0           0.0
46     MG02        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed  5.902190e+07  5.902190e+07          31         31 2021-03-25 18:00:00+08:00 2021-03-25 18:00:00+08:00  5.902190e+07  5.902190e+07            0.0           0.0
124    MG02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies  1.000000e+00  1.000000e+00          31         31 2021-03-25 18:00:00+08:00 2021-03-25 18:00:00+08:00  1.000000e+00  1.000000e+00            0.0           0.0
127    MG02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...  3.000000e-01  3.000000e-01          31         31 2021-03-25 18:00:00+08:00 2021-03-25 18:00:00+08:00  3.000000e-01  3.000000e-01            0.0           0.0
128    MG02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount  3.000000e+00  3.000000e+00          31         31 2021-03-25 18:00:00+08:00 2021-03-25 18:00:00+08:00  3.000000e+00  3.000000e+00            0.0           0.0
31     MG01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT  0.000000e+00  0.000000e+00          31         30 2021-03-25 18:00:00+08:00 2021-03-25 18:00:00+08:00  1.000000e+00  0.000000e+00            NaN           NaN
34     MG01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies  2.000000e+00  2.000000e+00          30         30 2021-03-25 18:00:00+08:00 2021-03-25 18:00:00+08:00  2.000000e+00  2.000000e+00            0.0           0.0
38     MG01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount  3.000000e+00  3.000000e+00          30         30 2021-03-25 18:00:00+08:00 2021-03-25 18:00:00+08:00  3.000000e+00  3.000000e+00            0.0           0.0
32     MG01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT  0.000000e+00  0.000000e+00          30         29 2021-03-25 18:00:00+08:00 2021-03-25 18:00:00+08:00  1.000000e+00  0.000000e+00            NaN           NaN
115    MG02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT  0.000000e+00  0.000000e+00          30         30 2021-03-25 18:00:00+08:00 2021-03-25 18:00:00+08:00  0.000000e+00  0.000000e+00            NaN           NaN
116    MG02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT  1.000000e+00  0.000000e+00           3         28 2021-03-25 18:01:00+08:00 2021-03-25 18:00:00+08:00  1.000000e+00  0.000000e+00            0.0           NaN
174    MG01       OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc  9.800000e+01  9.800000e+01          21         21 2021-03-25 18:01:00+08:00 2021-03-25 18:01:00+08:00  9.800000e+01  9.800000e+01            0.0           0.0
186    MG02       OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc  9.800000e+01  9.800000e+01          20         20 2021-03-25 18:01:00+08:00 2021-03-25 18:01:00+08:00  9.800000e+01  9.800000e+01            0.0           0.0
35     MG01          OSLinux-OSLinux_PROCESS_PROCESS_PROCPPMem  5.800000e+01  5.800000e+01          19         19 2021-03-25 18:00:00+08:00 2021-03-25 18:00:00+08:00  5.800000e+01  5.800000e+01            0.0           0.0
125    MG02          OSLinux-OSLinux_PROCESS_PROCESS_PROCPPMem  5.800000e+01  5.800000e+01          19         19 2021-03-25 18:00:00+08:00 2021-03-25 18:00:00+08:00  5.800000e+01  5.800000e+01            0.0           0.0
36     MG01      OSLinux-OSLinux_PROCESS_PROCESS_PROCPPMemPerc  0.000000e+00  0.000000e+00          16         16 2021-03-25 18:00:00+08:00 2021-03-25 18:00:00+08:00  0.000000e+00  0.000000e+00            NaN           NaN
39     MG01    OSLinux-OSLinux_SWAP_SWAP_SWPTotSwapUsedPercent  0.000000e+00  0.000000e+00          16         16 2021-03-25 18:00:00+08:00 2021-03-25 18:00:00+08:00  0.000000e+00  0.000000e+00            NaN           NaN, 209)```
```

In [ ]:
# In[7]:

The Executor failed to complete the instruction, please re-write a new instruction for Executor.

```
Out[7]:
```
The Executor failed to complete the instruction, please re-write a new instruction for Executor.```
```

In [ ]:
# In[8]:

The Executor failed to complete the instruction, please re-write a new instruction for Executor.

```
Out[8]:
```
The Executor failed to complete the instruction, please re-write a new instruction for Executor.```
```

In [ ]:
# In[9]:

import pandas as pd
import pytz
from pathlib import Path

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing log_df if present, otherwise load
if 'log_df' in globals():
    df = log_df
else:
    path = Path("log.csv")
    df = pd.read_csv(path) if path.exists() else pd.DataFrame(columns=["timestamp","cmdb_id","log_name","value"])

# Ensure timestamp/dt column exists and is timezone-aware (UTC+8)
if 'dt' not in df.columns or df['dt'].isna().all():
    if 'timestamp' in df.columns and not df.empty:
        df['dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        df['dt'] = pd.Series(dtype='datetime64[ns]')

# Define filter criteria
targets = ['MG01', 'MG02']
start = pd.to_datetime('2021-03-25 18:00:00').tz_localize(tz)
end = pd.to_datetime('2021-03-25 18:30:00').tz_localize(tz)

# 1) Filter log rows for MG01/MG02 within the window (inclusive)
log_mg_window = df[
    df['cmdb_id'].isin(targets) &
    (df['dt'] >= start) &
    (df['dt'] <= end)
].copy()

# 2) Prepare bounded display: up to 50 rows sorted by timestamp ascending, include requested columns
# Ensure original numeric timestamp column exists; if not, create from dt
if 'timestamp' not in log_mg_window.columns:
    # convert dt back to epoch seconds
    log_mg_window['timestamp'] = (log_mg_window['dt'].astype('int64') // 10**9).astype('int')

log_mg_window_sorted = log_mg_window.sort_values('dt', ascending=True)
log_mg_window_head = log_mg_window_sorted[['timestamp','cmdb_id','log_name','value']].head(50)

# 3) Small summary: total returned rows, counts per cmdb_id, unique log_name list (limit 20)
total_rows_returned = int(log_mg_window.shape[0])
counts_per_cmdb = log_mg_window['cmdb_id'].value_counts().reindex(targets).fillna(0).astype(int)
unique_log_names = sorted(log_mg_window['log_name'].dropna().unique().tolist())[:20]

summary_df = pd.DataFrame([{
    'total_rows_returned': total_rows_returned,
    'count_MG01': int(counts_per_cmdb.get('MG01', 0)),
    'count_MG02': int(counts_per_cmdb.get('MG02', 0)),
    'unique_log_names_sample': unique_log_names
}])

# Keep full filtered table in kernel variable `log_mg_window` (already defined)
# Display compact bounded outputs: head (<=50 rows) and the single-row summary dataframe
log_mg_window_head, summary_df

```
Out[9]:
```
```python
# Build a concise plain-English summary from the kernel results (log_mg_window_head, summary_df, log_mg_window).
# The full filtered table is stored in `log_mg_window`.

import pandas as pd

# Safely read summary_df if available
if 'summary_df' in globals() and not summary_df.empty:
    row = summary_df.iloc[0]
    total = int(row.get('total_rows_returned', 0))
    mg01 = int(row.get('count_MG01', 0))
    mg02 = int(row.get('count_MG02', 0))
    names = row.get('unique_log_names_sample', []) or []
else:
    total = int(log_mg_window.shape[0]) if 'log_mg_window' in globals() else 0
    mg01 = int((log_mg_window['cmdb_id']=='MG01').sum()) if 'log_mg_window' in globals() else 0
    mg02 = int((log_mg_window['cmdb_id']=='MG02').sum()) if 'log_mg_window' in globals() else 0
    names = []

summary = (
    f"Log inspection (MG01 & MG02, 2021-03-25 18:00:00+08:00 to 2021-03-25 18:30:00+08:00): "
    f"Total rows returned = {total}; counts: MG01={mg01}, MG02={mg02}; "
    f"Unique log_name sample (<=20) = {names}. "
    f"Full filtered table is stored in the kernel variable 'log_mg_window'."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [timestamp, cmdb_id, log_name, value]
Index: [],    total_rows_returned  count_MG01  count_MG02 unique_log_names_sample
0                    0           0           0                      [])```
```

In [ ]:
# In[10]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing kernel variables where available
# metric_mg_window: filtered metrics for MG01/MG02 within the incident window
# metric_mg_thresholds: global p95/p5 thresholds for MG01/MG02 computed from full series

if 'metric_mg_window' not in globals():
    # fallback: try to derive metric_mg_window from metric_df (should exist)
    if 'metric_df' in globals():
        df = metric_df
        # ensure dt column
        if 'dt' not in df.columns or df['dt'].isna().all():
            df['dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
        start = pd.to_datetime('2021-03-25 18:00:00').tz_localize(tz)
        end = pd.to_datetime('2021-03-25 18:30:00').tz_localize(tz)
        metric_mg_window = df[df['cmdb_id'].isin(['MG01','MG02']) & (df['dt'] >= start) & (df['dt'] <= end)].copy()
    else:
        metric_mg_window = pd.DataFrame(columns=['timestamp','cmdb_id','kpi_name','value','dt'])

# Ensure dt exists and is timezone-aware
if 'dt' not in metric_mg_window.columns or metric_mg_window['dt'].isna().all():
    if 'timestamp' in metric_mg_window.columns and not metric_mg_window.empty:
        metric_mg_window['dt'] = pd.to_datetime(metric_mg_window['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        metric_mg_window['dt'] = pd.Series(dtype='datetime64[ns]')

# Ensure thresholds exist; if not, compute from full metric_df (global before filtering)
if 'metric_mg_thresholds' not in globals():
    if 'metric_df' in globals():
        metric_mg_thresholds = (
            metric_df[metric_df['cmdb_id'].isin(['MG01','MG02'])]
            .groupby(['cmdb_id','kpi_name'])['value']
            .agg(p95=lambda s: float(s.quantile(0.95)), p5=lambda s: float(s.quantile(0.05)))
            .reset_index()
        )
    else:
        metric_mg_thresholds = pd.DataFrame(columns=['cmdb_id','kpi_name','p95','p5'])

# 1) Filter metric_mg_window to kpi_name containing 'MEM' (case-insensitive)
mem_mask = metric_mg_window['kpi_name'].astype(str).str.contains('mem', case=False, na=False)
metric_mg_mem_window = metric_mg_window[mem_mask].copy()

# Prepare bounded display: up to 50 rows sorted by timestamp ascending with requested columns
# Ensure a numeric timestamp column exists
if 'timestamp' not in metric_mg_mem_window.columns and 'dt' in metric_mg_mem_window.columns:
    metric_mg_mem_window['timestamp'] = (metric_mg_mem_window['dt'].astype('int64') // 10**9).astype('int')

metric_mg_mem_window_sorted = metric_mg_mem_window.sort_values('dt', ascending=True)
metric_mg_mem_window_head = metric_mg_mem_window_sorted[['timestamp','cmdb_id','kpi_name','value']].head(50)

# 2) Group summaries per (cmdb_id, kpi_name) for this filtered set (limit top 20 by window_max_value desc)
if metric_mg_mem_window.empty:
    mem_group_summary = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','global_p95','window_max_value','window_min_value',
        'earliest_ts_value_ge_global_p95','earliest_ts_of_window_max'
    ])
else:
    # merge global p95 into window rows
    mm = metric_mg_mem_window.merge(metric_mg_thresholds[['cmdb_id','kpi_name','p95']], on=['cmdb_id','kpi_name'], how='left')
    mm.rename(columns={'p95':'global_p95'}, inplace=True)

    # compute window extremes per group
    extremes = mm.groupby(['cmdb_id','kpi_name'])['value'].agg(window_max_value='max', window_min_value='min').reset_index()

    # earliest timestamp within window where value >= global_p95
    ge_p95 = mm[mm['value'] >= mm['global_p95']]
    earliest_ge = ge_p95.groupby(['cmdb_id','kpi_name'])['dt'].min().reset_index().rename(columns={'dt':'earliest_ts_value_ge_global_p95'})

    # earliest timestamp within window where value == window_max_value
    # need to join extremes back to rows to find rows equal to max, then pick earliest dt
    mm_with_max = mm.merge(extremes[['cmdb_id','kpi_name','window_max_value']], on=['cmdb_id','kpi_name'], how='left')
    is_max = mm_with_max[mm_with_max['value'] == mm_with_max['window_max_value']]
    earliest_max = is_max.groupby(['cmdb_id','kpi_name'])['dt'].min().reset_index().rename(columns={'dt':'earliest_ts_of_window_max'})

    # Build summary
    mem_group_summary = (
        extremes
        .merge(metric_mg_thresholds[['cmdb_id','kpi_name','p95']], on=['cmdb_id','kpi_name'], how='left')
        .rename(columns={'p95':'global_p95'})
        .merge(earliest_ge, on=['cmdb_id','kpi_name'], how='left')
        .merge(earliest_max, on=['cmdb_id','kpi_name'], how='left')
    )

    # Ensure timestamp columns are timezone-aware datetimes or NaT
    mem_group_summary['earliest_ts_value_ge_global_p95'] = pd.to_datetime(mem_group_summary['earliest_ts_value_ge_global_p95'], errors='coerce')
    mem_group_summary['earliest_ts_of_window_max'] = pd.to_datetime(mem_group_summary['earliest_ts_of_window_max'], errors='coerce')
    # Localize naive timestamps to tz if needed
    for col in ['earliest_ts_value_ge_global_p95','earliest_ts_of_window_max']:
        if not mem_group_summary.empty and mem_group_summary[col].dt.tz is None:
            try:
                mem_group_summary[col] = mem_group_summary[col].dt.tz_localize(tz)
            except Exception:
                # if already tz-aware or cannot localize, attempt convert
                mem_group_summary[col] = mem_group_summary[col].dt.tz_convert(tz)

    # Sort and limit to top 20 by window_max_value desc
    mem_group_summary = mem_group_summary.sort_values('window_max_value', ascending=False).head(20)

# 3) Earliest timestamp across all these memory groups where value >= corresponding global_p95
if not mem_group_summary.empty and mem_group_summary['earliest_ts_value_ge_global_p95'].notna().any():
    earliest_cross_group_ts = mem_group_summary['earliest_ts_value_ge_global_p95'].min()
else:
    earliest_cross_group_ts = pd.NaT

# Store full intermediate results in kernel variables (already assigned)
mem_window = metric_mg_mem_window        # full filtered set (may be larger than head)
mem_group_summary_full = mem_group_summary

# Display compact bounded outputs: mem_window_head (<=50 rows), mem_group_summary (<=20 groups), earliest_cross_group_ts
metric_mg_mem_window_head, mem_group_summary_full, earliest_cross_group_ts

```
Out[10]:
```
```python
# Build a concise plain-English summary from the memory-group results already in the kernel.
# Uses existing variables: mem_window, mem_group_summary_full, earliest_cross_group_ts (if present).

import pandas as pd

# Safely access kernel variables
mem_window = globals().get('mem_window', pd.DataFrame())
mem_group_summary = globals().get('mem_group_summary_full', pd.DataFrame())
earliest_cross = globals().get('earliest_cross_group_ts', pd.NaT)

# Counts
num_mem_rows = int(mem_window.shape[0]) if not mem_window.empty else 0
num_mem_groups = int(mem_group_summary.shape[0]) if not mem_group_summary.empty else 0

# Top up to 3 example groups by window_max_value
examples = []
if not mem_group_summary.empty:
    ex_rows = mem_group_summary.head(3)
    for _, r in ex_rows.iterrows():
        cp = r.get('cmdb_id','')
        kn = r.get('kpi_name','')
        wmax = r.get('window_max_value')
        gp95 = r.get('global_p95')
        eht = r.get('earliest_ts_value_ge_global_p95')
        eht_str = eht.isoformat() if pd.notna(eht) else ''
        examples.append(f"{cp} | {kn} | window_max={wmax} | global_p95={gp95} | first_ge_p95={eht_str}")

# Compose summary string
summary_lines = [
    "Memory-metric inspection (MG01 & MG02, 2021-03-25 18:00:00+08:00 to 18:30:00+08:00):",
    f"- Total memory-related metric rows in window: {num_mem_rows}",
    f"- Distinct memory metric groups examined: {num_mem_groups}",
    f"- Earliest timestamp when any memory metric reached or exceeded its global p95: {earliest_cross if pd.notna(earliest_cross) else 'None'}",
    "- Top example memory groups (up to 3):"
]
if examples:
    for e in examples:
        summary_lines.append(f"  * {e}")
else:
    summary_lines.append("  * (no memory groups found)")

summary = "\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(       timestamp cmdb_id                                       kpi_name         value
85    1616666400    MG01       JVM-Memory_7779_JVM_Memory_HeapMemoryMax  2.040136e+09
211   1616666400    MG02  OSLinux-OSLinux_PROCESS_PROCESS_PROCPPMemPerc  0.000000e+00
210   1616666400    MG02      OSLinux-OSLinux_PROCESS_PROCESS_PROCPPMem  5.800000e+01
199   1616666400    MG02          OSLinux-OSLinux_MEMORY_MEMORY_UserMem  7.671000e+03
198   1616666400    MG02       OSLinux-OSLinux_MEMORY_MEMORY_MEMFreeMem  1.170000e+02
...          ...     ...                                            ...           ...
1781  1616666520    MG01       JVM-Memory_7779_JVM_Memory_HeapMemoryMax  2.040136e+09
1809  1616666520    MG01   OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc  9.800000e+01
1810  1616666520    MG01   OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc  2.798820e+01
1811  1616666520    MG01          OSLinux-OSLinux_MEMORY_MEMORY_UserMem  7.661000e+03
1808  1616666520    MG01       OSLinux-OSLinux_MEMORY_MEMORY_MEMFreeMem  1.280000e+02

[50 rows x 4 columns],    cmdb_id                                      kpi_name  window_max_value  window_min_value    global_p95 earliest_ts_value_ge_global_p95 earliest_ts_of_window_max
0     MG01      JVM-Memory_7779_JVM_Memory_HeapMemoryMax      2.040136e+09      2.040136e+09  2.040136e+09       2021-03-25 18:00:00+08:00 2021-03-25 18:00:00+08:00
11    MG02      JVM-Memory_7779_JVM_Memory_HeapMemoryMax      2.040136e+09      2.040136e+09  2.040136e+09       2021-03-25 18:00:00+08:00 2021-03-25 18:00:00+08:00
2     MG01     JVM-Memory_7779_JVM_Memory_HeapMemoryUsed      1.033511e+09      2.119431e+08  1.006211e+09       2021-03-25 18:15:00+08:00 2021-03-25 18:15:00+08:00
13    MG02     JVM-Memory_7779_JVM_Memory_HeapMemoryUsed      9.239938e+08      2.264370e+08  9.148955e+08       2021-03-25 18:14:00+08:00 2021-03-25 18:14:00+08:00
14    MG02   JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed      5.902190e+07      5.902190e+07  5.902190e+07       2021-03-25 18:00:00+08:00 2021-03-25 18:00:00+08:00
3     MG01   JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed      5.886139e+07      5.886139e+07  5.886139e+07       2021-03-25 18:00:00+08:00 2021-03-25 18:00:00+08:00
19    MG02         OSLinux-OSLinux_MEMORY_MEMORY_UserMem      7.672000e+03      7.658000e+03  7.671100e+03       2021-03-25 18:09:00+08:00 2021-03-25 18:09:00+08:00
8     MG01         OSLinux-OSLinux_MEMORY_MEMORY_UserMem      7.672000e+03      7.652000e+03  7.671050e+03       2021-03-25 18:07:00+08:00 2021-03-25 18:07:00+08:00
15    MG02        OSLinux-OSLinux_MEMORY_MEMORY_CacheMem      5.537000e+03      5.522000e+03  5.536100e+03       2021-03-25 18:00:00+08:00 2021-03-25 18:00:00+08:00
4     MG01        OSLinux-OSLinux_MEMORY_MEMORY_CacheMem      5.494000e+03      5.475000e+03  5.492200e+03       2021-03-25 18:27:00+08:00 2021-03-25 18:27:00+08:00
5     MG01      OSLinux-OSLinux_MEMORY_MEMORY_MEMFreeMem      1.360000e+02      1.170000e+02  1.351000e+02       2021-03-25 18:28:00+08:00 2021-03-25 18:28:00+08:00
16    MG02      OSLinux-OSLinux_MEMORY_MEMORY_MEMFreeMem      1.310000e+02      1.170000e+02  1.310000e+02       2021-03-25 18:27:00+08:00 2021-03-25 18:27:00+08:00
17    MG02  OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc      9.800000e+01      9.800000e+01  9.800000e+01       2021-03-25 18:01:00+08:00 2021-03-25 18:01:00+08:00
6     MG01  OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc      9.800000e+01      9.800000e+01  9.800000e+01       2021-03-25 18:01:00+08:00 2021-03-25 18:01:00+08:00
9     MG01     OSLinux-OSLinux_PROCESS_PROCESS_PROCPPMem      5.800000e+01      5.800000e+01  5.800000e+01       2021-03-25 18:00:00+08:00 2021-03-25 18:00:00+08:00
20    MG02     OSLinux-OSLinux_PROCESS_PROCESS_PROCPPMem      5.800000e+01      5.800000e+01  5.800000e+01       2021-03-25 18:00:00+08:00 2021-03-25 18:00:00+08:00
1     MG01    JVM-Memory_7779_JVM_Memory_HeapMemoryUsage      4.835020e+01      1.038870e+01  4.826727e+01       2021-03-25 18:20:00+08:00 2021-03-25 18:20:00+08:00
12    MG02    JVM-Memory_7779_JVM_Memory_HeapMemoryUsage      4.529080e+01      1.200030e+01  4.410568e+01       2021-03-25 18:15:00+08:00 2021-03-25 18:15:00+08:00
7     MG01  OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc      2.800100e+01      2.793680e+01  2.800100e+01       2021-03-25 18:08:00+08:00 2021-03-25 18:08:00+08:00
18    MG02  OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc      2.743610e+01      2.739760e+01  2.743610e+01       2021-03-25 18:20:00+08:00 2021-03-25 18:20:00+08:00, Timestamp('2021-03-25 18:00:00+0800', tz='Asia/Shanghai'))

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[11]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse metric_df from kernel
if 'metric_df' not in globals():
    metric_df = pd.read_csv("metric.csv")

# Ensure dt column exists and is timezone-aware Asia/Shanghai
if 'dt' not in metric_df.columns or metric_df['dt'].isna().all():
    if 'timestamp' in metric_df.columns and not metric_df.empty:
        metric_df['dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        metric_df['dt'] = pd.Series(dtype='datetime64[ns]')

# Target components and window
targets = ['MG01', 'MG02']
start = pd.to_datetime('2021-03-25 18:00:00').tz_localize(tz)
end = pd.to_datetime('2021-03-25 18:30:00').tz_localize(tz)

# 1) Identify memory-related KPI names (case-insensitive 'mem' or 'memory') across the full metric series
metric_df['kpi_name_str'] = metric_df['kpi_name'].astype(str)
mem_mask_full = metric_df['kpi_name_str'].str.contains('mem', case=False, na=False)
metric_mem_full = metric_df[mem_mask_full].copy()

# Compute global p95 per (cmdb_id, kpi_name) using the entire series BEFORE filtering
if not metric_mem_full.empty:
    metric_mem_thresholds = (
        metric_mem_full
        .groupby(['cmdb_id','kpi_name'])['value']
        .agg(global_p95=lambda s: float(s.quantile(0.95)))
        .reset_index()
    )
else:
    metric_mem_thresholds = pd.DataFrame(columns=['cmdb_id','kpi_name','global_p95'])

# 2) Filter metric rows to incident window and targets and memory KPIs
metric_mem_window = metric_df[
    metric_df['cmdb_id'].isin(targets) &
    mem_mask_full &
    (metric_df['dt'] >= start) &
    (metric_df['dt'] <= end)
].copy()

# Ensure timestamp column exists (epoch seconds) for compact output
if 'timestamp' not in metric_mem_window.columns:
    metric_mem_window['timestamp'] = (metric_mem_window['dt'].astype('int64') // 10**9).astype('int')

# 3) Per-(cmdb_id,kpi_name) summary within window, merging global_p95
if metric_mem_window.empty:
    metric_mem_kpi_summary = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','global_p95','window_max_value','window_min_value',
        'high_count','earliest_high_ts','severity_high'
    ])
else:
    mm = metric_mem_window.merge(metric_mem_thresholds, on=['cmdb_id','kpi_name'], how='left')
    # compute window extremes and counts
    extremes = mm.groupby(['cmdb_id','kpi_name'])['value'].agg(window_max_value='max', window_min_value='min').reset_index()
    counts = mm.assign(is_high = mm['value'] >= mm['global_p95']).groupby(['cmdb_id','kpi_name'])['is_high'].sum().reset_index().rename(columns={'is_high':'high_count'})
    # earliest timestamp in window where value >= global_p95
    ge = mm[mm['value'] >= mm['global_p95']]
    earliest_ge = ge.groupby(['cmdb_id','kpi_name'])['dt'].min().reset_index().rename(columns={'dt':'earliest_high_ts'})
    # merge
    metric_mem_kpi_summary = (
        extremes
        .merge(metric_mem_thresholds, on=['cmdb_id','kpi_name'], how='left')
        .merge(counts, on=['cmdb_id','kpi_name'], how='left')
        .merge(earliest_ge, on=['cmdb_id','kpi_name'], how='left')
    )
    metric_mem_kpi_summary['high_count'] = metric_mem_kpi_summary['high_count'].fillna(0).astype(int)
    # compute severity_high safely
    def comp_sev(row):
        gp = row['global_p95']
        if row['high_count'] > 0 and pd.notna(gp) and gp != 0:
            return float((row['window_max_value'] - gp) / gp)
        return np.nan
    metric_mem_kpi_summary['severity_high'] = metric_mem_kpi_summary.apply(comp_sev, axis=1)

# 4A) Per-KPI table: top 20 rows sorted by severity_high desc (NaNs last), show requested columns
metric_mem_kpi_summary_sorted = metric_mem_kpi_summary.copy()
metric_mem_kpi_summary_sorted['sev_sort'] = metric_mem_kpi_summary_sorted['severity_high'].fillna(-np.inf)
metric_mem_kpi_summary_sorted = metric_mem_kpi_summary_sorted.sort_values(['sev_sort','window_max_value'], ascending=[False, False])
metric_mem_kpi_summary_top20 = metric_mem_kpi_summary_sorted[[
    'cmdb_id','kpi_name','global_p95','window_max_value','high_count','earliest_high_ts','severity_high'
]].head(20)

# 4B) Per-component aggregated summary for MG01 and MG02
if metric_mem_kpi_summary.empty:
    mem_component_summary = pd.DataFrame(columns=[
        'cmdb_id','num_memory_kpis_examined','num_kpis_with_high','total_high_count',
        'earliest_high_ts_across_kpis','max_severity_high_across_kpis'
    ])
else:
    comp = metric_mem_kpi_summary.copy()
    agg = comp.groupby('cmdb_id').agg(
        num_memory_kpis_examined = ('kpi_name','nunique'),
        num_kpis_with_high = ('high_count', lambda s: int((s>0).sum())),
        total_high_count = ('high_count','sum'),
        earliest_high_ts_across_kpis = ('earliest_high_ts', lambda s: s.min() if s.notna().any() else pd.NaT),
        max_severity_high_across_kpis = ('severity_high', lambda s: float(s.dropna().max()) if s.dropna().size>0 else np.nan)
    ).reset_index()
    # Keep only MG01 and MG02 rows (ensure both present)
    mem_component_summary = agg[agg['cmdb_id'].isin(targets)].copy()

# 5) Scalar: earliest_high_ts_across_both_components (min earliest_high_ts across MG01 & MG02)
if not mem_component_summary.empty and mem_component_summary['earliest_high_ts_across_kpis'].notna().any():
    earliest_high_ts_across_both = mem_component_summary['earliest_high_ts_across_kpis'].min()
else:
    earliest_high_ts_across_both = pd.NaT

# Store full intermediate tables in kernel variables (already stored): metric_mem_thresholds, metric_mem_window, metric_mem_kpi_summary, mem_component_summary

# Final compact outputs (bounded): metric_mem_kpi_summary_top20 (<=20 rows), mem_component_summary (<=2 rows), earliest_high_ts_across_both
metric_mem_kpi_summary_top20, mem_component_summary, earliest_high_ts_across_both

```
Out[11]:
```
```python
# Compose a concise plain-English summary from kernel results and display as a single string.
# Uses existing variables: metric_mem_kpi_summary_top20, mem_component_summary, earliest_high_ts_across_both.

from datetime import datetime
import pandas as pd

top_kpi = metric_mem_kpi_summary_top20 if 'metric_mem_kpi_summary_top20' in globals() else pd.DataFrame()
comp = mem_component_summary if 'mem_component_summary' in globals() else pd.DataFrame()
earliest_both = earliest_high_ts_across_both if 'earliest_high_ts_across_both' in globals() else pd.NaT

lines = []
lines.append("Memory-focused metric summary (MG01 & MG02) — window 2021-03-25 18:00:00+08:00 to 18:30:00+08:00:")
lines.append("")
# Per-KPI top examples
if top_kpi is not None and not top_kpi.empty:
    lines.append(f"- Top memory KPIs by severity_high (showing up to {min(20, len(top_kpi))}):")
    for _, r in top_kpi.head(5).iterrows():  # show up to 5 compact examples
        cid = r['cmdb_id']
        k = r['kpi_name']
        gp = r['global_p95']
        wmax = r['window_max_value']
        hc = int(r['high_count'])
        eht = r['earliest_high_ts']
        eht_str = eht.isoformat() if pd.notna(eht) else 'None'
        sev = r['severity_high']
        sev_str = f"{sev:.6g}" if pd.notna(sev) else 'None'
        lines.append(f"  * {cid} | {k} — severity_high={sev_str}, high_count={hc}, earliest_high={eht_str}, global_p95={gp}, window_max={wmax}")
else:
    lines.append("- No memory KPI rows found in the window.")

lines.append("")
# Per-component summary
if comp is not None and not comp.empty:
    lines.append("- Per-component aggregated summary:")
    for _, r in comp.iterrows():
        cid = r['cmdb_id']
        num_kpis = int(r['num_memory_kpis_examined'])
        num_with_high = int(r['num_kpis_with_high'])
        total_hc = int(r['total_high_count'])
        eht_comp = r['earliest_high_ts_across_kpis']
        ehtc_str = eht_comp.isoformat() if pd.notna(eht_comp) else 'None'
        max_sev = r['max_severity_high_across_kpis']
        max_sev_str = f"{max_sev:.6g}" if pd.notna(max_sev) else 'None'
        lines.append(f"  * {cid}: examined_kpis={num_kpis}, kpis_with_high={num_with_high}, total_high_count={total_hc}, earliest_high={ehtc_str}, max_severity={max_sev_str}")
else:
    lines.append("- No per-component summary available.")

lines.append("")
earliest_both_str = earliest_both.isoformat() if pd.notna(earliest_both) else 'None'
lines.append(f"- Earliest timestamp when any memory KPI >= global p95 across both components: {earliest_both_str}")
lines.append("")
lines.append("Notes: global p95 values were computed from the full metric series before filtering the window. Full intermediate tables are stored in kernel variables: metric_mem_thresholds, metric_mem_window, metric_mem_kpi_summary, mem_component_summary.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id                                      kpi_name    global_p95  window_max_value  high_count          earliest_high_ts  severity_high
2     MG01     JVM-Memory_7779_JVM_Memory_HeapMemoryUsed  1.006211e+09      1.033511e+09           2 2021-03-25 18:15:00+08:00       0.027132
12    MG02    JVM-Memory_7779_JVM_Memory_HeapMemoryUsage  4.410568e+01      4.529080e+01           1 2021-03-25 18:15:00+08:00       0.026870
13    MG02     JVM-Memory_7779_JVM_Memory_HeapMemoryUsed  9.148955e+08      9.239938e+08           2 2021-03-25 18:14:00+08:00       0.009945
5     MG01      OSLinux-OSLinux_MEMORY_MEMORY_MEMFreeMem  1.351000e+02      1.360000e+02           1 2021-03-25 18:28:00+08:00       0.006662
1     MG01    JVM-Memory_7779_JVM_Memory_HeapMemoryUsage  4.826727e+01      4.835020e+01           1 2021-03-25 18:20:00+08:00       0.001718
4     MG01        OSLinux-OSLinux_MEMORY_MEMORY_CacheMem  5.492200e+03      5.494000e+03           1 2021-03-25 18:27:00+08:00       0.000328
15    MG02        OSLinux-OSLinux_MEMORY_MEMORY_CacheMem  5.536100e+03      5.537000e+03           1 2021-03-25 18:00:00+08:00       0.000163
8     MG01         OSLinux-OSLinux_MEMORY_MEMORY_UserMem  7.671050e+03      7.672000e+03           1 2021-03-25 18:07:00+08:00       0.000124
19    MG02         OSLinux-OSLinux_MEMORY_MEMORY_UserMem  7.671100e+03      7.672000e+03           1 2021-03-25 18:09:00+08:00       0.000117
0     MG01      JVM-Memory_7779_JVM_Memory_HeapMemoryMax  2.040136e+09      2.040136e+09          31 2021-03-25 18:00:00+08:00       0.000000
11    MG02      JVM-Memory_7779_JVM_Memory_HeapMemoryMax  2.040136e+09      2.040136e+09          31 2021-03-25 18:00:00+08:00       0.000000
14    MG02   JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed  5.902190e+07      5.902190e+07          31 2021-03-25 18:00:00+08:00       0.000000
3     MG01   JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed  5.886139e+07      5.886139e+07          31 2021-03-25 18:00:00+08:00       0.000000
16    MG02      OSLinux-OSLinux_MEMORY_MEMORY_MEMFreeMem  1.310000e+02      1.310000e+02           2 2021-03-25 18:27:00+08:00       0.000000
6     MG01  OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc  9.800000e+01      9.800000e+01          21 2021-03-25 18:01:00+08:00       0.000000
17    MG02  OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc  9.800000e+01      9.800000e+01          20 2021-03-25 18:01:00+08:00       0.000000
9     MG01     OSLinux-OSLinux_PROCESS_PROCESS_PROCPPMem  5.800000e+01      5.800000e+01          19 2021-03-25 18:00:00+08:00       0.000000
20    MG02     OSLinux-OSLinux_PROCESS_PROCESS_PROCPPMem  5.800000e+01      5.800000e+01          19 2021-03-25 18:00:00+08:00       0.000000
7     MG01  OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc  2.800100e+01      2.800100e+01           3 2021-03-25 18:08:00+08:00       0.000000
18    MG02  OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc  2.743610e+01      2.743610e+01           3 2021-03-25 18:20:00+08:00       0.000000,   cmdb_id  num_memory_kpis_examined  num_kpis_with_high  total_high_count earliest_high_ts_across_kpis  max_severity_high_across_kpis
0    MG01                        11                  11               127    2021-03-25 18:00:00+08:00                       0.027132
1    MG02                        11                  11               127    2021-03-25 18:00:00+08:00                       0.026870, Timestamp('2021-03-25 18:00:00+0800', tz='Asia/Shanghai'))```
```

In [ ]:
# In[12]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse anomaly tables from kernel if available, with fallbacks
trace_tbl = None
for name in ('trace_anomaly_summary', 'trace_anomaly_summary_full', 'trace_anomaly_summary_top20'):
    if name in globals():
        trace_tbl = globals()[name].copy()
        break
if trace_tbl is None:
    trace_tbl = pd.DataFrame(columns=['cmdb_id','trace_name','high_count','severity_high','earliest_high_ts'])

mem_tbl = None
for name in ('metric_mem_kpi_summary', 'metric_mem_kpi_summary_full', 'metric_mem_kpi_summary_top20', 'metric_mg_anomaly'):
    if name in globals():
        mem_tbl = globals()[name].copy()
        break
if mem_tbl is None:
    mem_tbl = pd.DataFrame(columns=['cmdb_id','kpi_name','high_count','severity_high','earliest_high_ts'])

# Ensure relevant columns exist
for df in (trace_tbl, mem_tbl):
    for col in ['cmdb_id','high_count','severity_high','earliest_high_ts']:
        if col not in df.columns:
            df[col] = np.nan

# Normalize earliest_high_ts to timezone-aware datetimes (Asia/Shanghai)
def normalize_dtcol(s):
    s = pd.to_datetime(s, errors='coerce')
    if not s.empty:
        # if tz-naive, localize; if tz-aware, convert
        if s.dt.tz is None:
            try:
                s = s.dt.tz_localize(tz)
            except Exception:
                # fallback: treat as naive and localize
                s = s.dt.tz_localize(tz)
        else:
            s = s.dt.tz_convert(tz)
    return s

trace_tbl['earliest_high_ts'] = normalize_dtcol(trace_tbl.get('earliest_high_ts'))
mem_tbl['earliest_high_ts'] = normalize_dtcol(mem_tbl.get('earliest_high_ts'))

# Ensure numeric types for high_count and severity_high
trace_tbl['high_count'] = pd.to_numeric(trace_tbl['high_count'], errors='coerce').fillna(0).astype(int)
mem_tbl['high_count'] = pd.to_numeric(mem_tbl['high_count'], errors='coerce').fillna(0).astype(int)
trace_tbl['severity_high'] = pd.to_numeric(trace_tbl['severity_high'], errors='coerce')
mem_tbl['severity_high'] = pd.to_numeric(mem_tbl['severity_high'], errors='coerce')

# Aggregate per-cmdb_id for each table
trace_agg = trace_tbl.groupby('cmdb_id').agg(
    earliest_high_ts_trace = ('earliest_high_ts', lambda s: s.min() if s.notna().any() else pd.NaT),
    total_high_count_trace = ('high_count', 'sum'),
    max_severity_high_trace = ('severity_high', lambda s: float(s.dropna().max()) if s.dropna().size>0 else np.nan)
).reset_index()

mem_agg = mem_tbl.groupby('cmdb_id').agg(
    earliest_high_ts_mem = ('earliest_high_ts', lambda s: s.min() if s.notna().any() else pd.NaT),
    total_high_count_mem = ('high_count', 'sum'),
    max_severity_high_mem = ('severity_high', lambda s: float(s.dropna().max()) if s.dropna().size>0 else np.nan)
).reset_index()

# Merge the two per-component aggregations
per_component = pd.merge(trace_agg, mem_agg, on='cmdb_id', how='outer')

# Fill missing numeric totals with 0
per_component['total_high_count_trace'] = per_component['total_high_count_trace'].fillna(0).astype(int)
per_component['total_high_count_mem'] = per_component['total_high_count_mem'].fillna(0).astype(int)

# Compute combined metrics
def min_ts(a, b):
    # handle pd.NaT
    if pd.isna(a) and pd.isna(b):
        return pd.NaT
    if pd.isna(a):
        return b
    if pd.isna(b):
        return a
    return min(a, b)

per_component['earliest_high_ts'] = per_component.apply(
    lambda r: min_ts(r['earliest_high_ts_trace'], r['earliest_high_ts_mem']), axis=1
)

per_component['total_high_count'] = per_component['total_high_count_trace'] + per_component['total_high_count_mem']

per_component['max_severity_high'] = per_component[['max_severity_high_trace','max_severity_high_mem']].max(axis=1, skipna=True)

# Keep only requested columns and sort
per_component_summary = per_component[[
    'cmdb_id','earliest_high_ts','total_high_count','max_severity_high'
]].copy()

# Sort by earliest_high_ts ascending (NaT last), then total_high_count desc
per_component_summary['earliest_sort'] = per_component_summary['earliest_high_ts'].apply(lambda t: pd.Timestamp.max if pd.isna(t) else t)
per_component_summary = per_component_summary.sort_values(by=['earliest_sort','total_high_count'], ascending=[True, False]).drop(columns=['earliest_sort'])

# Limit to top 20 rows
per_component_summary_top20 = per_component_summary.head(20).reset_index(drop=True)

# Overall earliest anomaly timestamp across all components
valid_earliest = per_component_summary['earliest_high_ts'].dropna()
if not valid_earliest.empty:
    overall_earliest_anomaly_ts = valid_earliest.min()
    # components that match this earliest timestamp
    components_at_overall_earliest = per_component_summary.loc[
        per_component_summary['earliest_high_ts'] == overall_earliest_anomaly_ts, 'cmdb_id'
    ].tolist()
else:
    overall_earliest_anomaly_ts = pd.NaT
    components_at_overall_earliest = []

# Store intermediate results in kernel variables
per_component_merged_full = per_component  # full merged table (kept)
per_component_summary_top20 = per_component_summary_top20  # final compact table

# Display compact bounded outputs: per_component_summary_top20, overall_earliest_anomaly_ts, components_at_overall_earliest
per_component_summary_top20, overall_earliest_anomaly_ts, components_at_overall_earliest

```
Out[12]:
```
```python
# Compose a concise plain-English summary from kernel results:
# Uses variables: per_component_summary_top20, overall_earliest_anomaly_ts, components_at_overall_earliest

pc = globals().get('per_component_summary_top20', None)
overall_ts = globals().get('overall_earliest_anomaly_ts', None)
components_at_overall = globals().get('components_at_overall_earliest', [])

# Prepare top 3 components by total_high_count for quick examples
examples = []
if pc is not None and not pc.empty:
    top_by_count = pc.sort_values('total_high_count', ascending=False).head(3)
    for _, r in top_by_count.iterrows():
        cid = r['cmdb_id']
        thc = int(r['total_high_count'])
        sev = r['max_severity_high']
        sev_str = f"{sev:.6g}" if not pd.isna(sev) else "None"
        examples.append(f"{cid} (total_high_count={thc}, max_severity={sev_str})")

# Format overall timestamp
overall_ts_str = overall_ts.isoformat() if (hasattr(overall_ts, 'isoformat')) and pd.notna(overall_ts) else 'None'

summary_lines = [
    f"Earliest anomaly timestamp (overall): {overall_ts_str}",
    f"Components with that earliest anomaly: {components_at_overall}",
    "",
    "Top example components by total_high_count:",
]
if examples:
    for ex in examples:
        summary_lines.append("  - " + ex)
else:
    summary_lines.append("  - (no component examples available)")

summary_lines.append("")
summary_lines.append("Note: earliest_high_ts is taken from trace and memory KPI anomaly summaries combined.")

summary = "\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id          earliest_high_ts  total_high_count  max_severity_high
0       MG02 2021-03-25 18:00:00+08:00               207           5.597203
1       MG01 2021-03-25 18:00:00+08:00               205           1.955180
2   dockerB1 2021-03-25 18:00:00+08:00                40           1.429977
3   dockerB2 2021-03-25 18:00:00+08:00                34           2.840164
4   Tomcat01 2021-03-25 18:00:00+08:00                33           0.552632
5   Tomcat02 2021-03-25 18:00:00+08:00                33           3.203881
6   Tomcat03 2021-03-25 18:00:00+08:00                33           1.477741
7       IG02 2021-03-25 18:00:00+08:00                31           0.272593
8   Tomcat04 2021-03-25 18:00:00+08:00                31           0.224292
9   dockerA1 2021-03-25 18:01:00+08:00                35         111.111111
10  dockerA2 2021-03-25 18:01:00+08:00                33          22.200000
11      IG01 2021-03-25 18:01:00+08:00                30           0.286695, Timestamp('2021-03-25 18:00:00+0800', tz='Asia/Shanghai'), ['MG02', 'MG01', 'dockerB1', 'dockerB2', 'Tomcat01', 'Tomcat02', 'Tomcat03', 'IG02', 'Tomcat04'])```
```